In [ ]:
# =========================
# Phase 1 — Dataset Intelligence
# Validation Analysis
# Vehicle Behaviour Analysis
# Temporal Analysis
# Violation Analysis
# =========================

import ast
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# -------------------------
# Config
# -------------------------
INPUT_CSV = "jan to may police violation_anonymized791b166.csv"
OUT_DIR = Path("phase1_outputs_1")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TOP_N = 25
EPS = 1e-9

# -------------------------
# Helpers
# -------------------------
def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def parse_listlike(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    s = str(value).strip()
    if not s:
        return []
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, (list, tuple)):
            return list(parsed)
        return [parsed]
    except Exception:
        if s.startswith("[") and s.endswith("]"):
            s = s[1:-1]
        parts = [p.strip().strip("'").strip('"') for p in s.split(",")]
        return [p for p in parts if p]

def week_start_monday(series):
    dt = pd.to_datetime(series, errors="coerce", utc=True).dt.tz_convert("Asia/Kolkata")
    week_start = dt.dt.normalize() - pd.to_timedelta(dt.dt.weekday, unit="D")
    return week_start.dt.tz_localize(None)

def safe_div(a, b):
    return a / (b + EPS)

def ensure_required_columns(df, cols):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

def save_plot(path):
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()

def severity_from_tags(tags):
    """
    Maps violation tags to a 1–5 severity score.
    Uses the highest applicable severity if multiple tags exist.
    """
    severity_map = {
        5: {
            "DOUBLE PARKING",
            "NEAR ROAD CROSSING",
            "NEAR TRAFFIC LIGHT",
            "NEAR TRAFFIC LIGHT / ZEBRA CROSSING",
            "NEAR ZEBRA CROSSING",
        },
        4: {
            "PARKING IN MAIN ROAD",
            "NEAR BUS STOP",
            "NEAR SCHOOL",
            "NEAR HOSPITAL",
            "OPPOSITE ANOTHER VEHICLE",
        },
        3: {
            "PARKING ON FOOTPATH",
        },
        2: {
            "WRONG PARKING",
            "PARKING OTHER THAN BUS STOP",
        },
        1: {
            "NO PARKING",
        },
    }

    if not tags:
        return 1

    normalized = [clean_text(t).upper() for t in tags]
    score = 1
    for s, vocab in severity_map.items():
        if any(tag in vocab for tag in normalized):
            score = max(score, s)
    return score

def dominant_tag(tags):
    if not tags:
        return ""
    return clean_text(tags[0]).upper()

# -------------------------
# Load
# -------------------------
df = pd.read_csv(INPUT_CSV, low_memory=False)

required_cols = [
    "id", "latitude", "longitude", "vehicle_number", "vehicle_type",
    "violation_type", "created_datetime", "police_station",
    "junction_name", "validation_status"
]
ensure_required_columns(df, required_cols)

for col in ["validation_status", "police_station", "junction_name", "vehicle_number", "vehicle_type"]:
    df[col] = df[col].astype("string")

# -------------------------
# Core parsing
# -------------------------
df["created_datetime_parsed"] = pd.to_datetime(df["created_datetime"], errors="coerce", utc=True)
df["created_datetime_ist"] = df["created_datetime_parsed"].dt.tz_convert("Asia/Kolkata")
df["week_start"] = week_start_monday(df["created_datetime"])
df["hour_ist"] = df["created_datetime_ist"].dt.hour
df["day_of_week"] = df["created_datetime_ist"].dt.day_name()
df["month"] = df["created_datetime_ist"].dt.month
df["is_weekend"] = df["day_of_week"].isin(["Saturday", "Sunday"]).astype(int)

df["violation_tags"] = df["violation_type"].apply(parse_listlike)
df["tag_count"] = df["violation_tags"].apply(len)
df["severity_score"] = df["violation_tags"].apply(severity_from_tags)
df["dominant_violation_tag"] = df["violation_tags"].apply(dominant_tag)

junction_clean = df["junction_name"].fillna("").astype(str).str.strip()
has_junction = junction_clean.ne("") & junction_clean.str.upper().ne("NO JUNCTION")
police_clean = df["police_station"].fillna("UNKNOWN").astype(str).str.strip()
police_clean = police_clean.where(police_clean.ne(""), "UNKNOWN")

df["hotspot_unit"] = np.where(
    has_junction,
    "JUNCTION::" + junction_clean,
    "POLICE_STATION::" + police_clean
)

validation_clean = df["validation_status"].fillna("").astype(str).str.lower()
df["validation_status_clean"] = validation_clean

# A simple numeric flag useful for later phases
status_score_map = {
    "approved": 1.0,
    "rejected": 0.0,
    "processing": 0.5,
    "duplicate": 0.25,
    "created1": 0.5,
}
df["validation_score"] = df["validation_status_clean"].map(status_score_map).fillna(0.5)

# Keep only rows with valid time and coordinates for analysis tables that need them
df_valid = df.dropna(subset=["created_datetime_parsed", "latitude", "longitude"]).copy()

# -------------------------
# 1) Validation Analysis
# -------------------------
validation_summary = (
    df["validation_status_clean"]
    .replace("", "missing")
    .value_counts(dropna=False)
    .rename_axis("validation_status")
    .reset_index(name="count")
)
validation_summary["percent"] = 100 * validation_summary["count"] / len(df)

approval_rate = safe_div(
    int((df["validation_status_clean"] == "approved").sum()),
    len(df)
)

validation_by_vehicle_type = (
    df.groupby("vehicle_type", dropna=False)
    .agg(
        total_records=("id", "size"),
        approved_records=("validation_status_clean", lambda s: (s == "approved").sum()),
        rejected_records=("validation_status_clean", lambda s: (s == "rejected").sum()),
        approved_rate=("validation_status_clean", lambda s: (s == "approved").mean()),
    )
    .reset_index()
    .sort_values("total_records", ascending=False)
)

validation_by_police_station = (
    df.groupby("police_station", dropna=False)
    .agg(
        total_records=("id", "size"),
        approved_rate=("validation_status_clean", lambda s: (s == "approved").mean()),
        rejected_rate=("validation_status_clean", lambda s: (s == "rejected").mean()),
    )
    .reset_index()
    .sort_values(["total_records", "approved_rate"], ascending=[False, False])
)

validation_by_hotspot = (
    df.groupby("hotspot_unit", dropna=False)
    .agg(
        total_records=("id", "size"),
        approved_rate=("validation_status_clean", lambda s: (s == "approved").mean()),
        rejected_rate=("validation_status_clean", lambda s: (s == "rejected").mean()),
        processed_rate=("validation_status_clean", lambda s: (s == "processing").mean()),
    )
    .reset_index()
    .sort_values("total_records", ascending=False)
)

# Validation uncertainty score: high when approval rate is low or mixed
validation_by_hotspot["validation_uncertainty"] = 1.0 - validation_by_hotspot["approved_rate"]
validation_by_vehicle_type["validation_uncertainty"] = 1.0 - validation_by_vehicle_type["approved_rate"]

# -------------------------
# 2) Vehicle Behaviour Analysis
# -------------------------
vehicle_summary = (
    df.groupby("vehicle_number", dropna=False)
    .agg(
        total_violations=("id", "size"),
        unique_hotspots=("hotspot_unit", "nunique"),
        unique_junctions=("junction_name", lambda s: s.fillna("").astype(str).replace("No Junction", "").nunique()),
        first_seen=("created_datetime_parsed", "min"),
        last_seen=("created_datetime_parsed", "max"),
        vehicle_type_mode=("vehicle_type", lambda s: s.mode().iloc[0] if not s.mode().empty else ""),
        approved_violations=("validation_status_clean", lambda s: (s == "approved").sum()),
    )
    .reset_index()
)

vehicle_summary["active_days"] = (
    df_valid.groupby("vehicle_number")["created_datetime_ist"]
    .apply(lambda s: s.dt.date.nunique())
    .reindex(vehicle_summary["vehicle_number"])
    .values
)

vehicle_summary["repeat_violation_flag"] = (vehicle_summary["total_violations"] >= 2).astype(int)
vehicle_summary["chronic_offender_flag"] = (vehicle_summary["total_violations"] >= 5).astype(int)

vehicle_summary["approval_rate"] = safe_div(
    vehicle_summary["approved_violations"],
    vehicle_summary["total_violations"]
)

vehicle_summary = vehicle_summary.sort_values(
    ["total_violations", "approved_violations", "unique_hotspots"],
    ascending=[False, False, False]
).reset_index(drop=True)

top_vehicle_offenders = vehicle_summary.head(TOP_N).copy()
chronic_offenders = vehicle_summary[vehicle_summary["chronic_offender_flag"] == 1].copy()

# Per vehicle-type behaviour
vehicle_type_summary = (
    df.groupby("vehicle_type", dropna=False)
    .agg(
        records=("id", "size"),
        unique_vehicles=("vehicle_number", "nunique"),
        mean_severity=("severity_score", "mean"),
        median_severity=("severity_score", "median"),
        approval_rate=("validation_status_clean", lambda s: (s == "approved").mean()),
    )
    .reset_index()
    .sort_values("records", ascending=False)
)

# -------------------------
# 3) Temporal Analysis
# -------------------------
hourly_distribution = (
    df_valid.groupby("hour_ist")
    .size()
    .reset_index(name="count")
    .sort_values("hour_ist")
)
daily_distribution = (
    df_valid.groupby("day_of_week")
    .size()
    .reindex(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"])
    .reset_index(name="count")
)
weekly_distribution = (
    df_valid.groupby("week_start")
    .size()
    .reset_index(name="count")
    .sort_values("week_start")
)
monthly_distribution = (
    df_valid.groupby("month")
    .size()
    .reset_index(name="count")
    .sort_values("month")
)

temporal_by_hour_vehicle_type = (
    df_valid.groupby(["hour_ist", "vehicle_type"])
    .size()
    .reset_index(name="count")
)

temporal_by_day_validation = (
    df_valid.groupby(["day_of_week", "validation_status_clean"])
    .size()
    .reset_index(name="count")
)

peak_hour = int(hourly_distribution.loc[hourly_distribution["count"].idxmax(), "hour_ist"]) if len(hourly_distribution) else None
peak_day = daily_distribution.loc[daily_distribution["count"].idxmax(), "day_of_week"] if len(daily_distribution) else None

# Trend by week: useful for later emerging hotspot stage
weekly_trend_by_hotspot = (
    df_valid.groupby(["hotspot_unit", "week_start"])
    .size()
    .reset_index(name="weekly_count")
    .sort_values(["hotspot_unit", "week_start"])
)

# -------------------------
# 4) Violation Analysis
# -------------------------
# Tag-level analysis
tag_rows = []
for _, row in df.iterrows():
    tags = row["violation_tags"]
    if not tags:
        tag_rows.append({"violation_tag": "UNKNOWN", "id": row["id"], "vehicle_number": row["vehicle_number"], "hotspot_unit": row["hotspot_unit"]})
    else:
        for t in tags:
            tag_rows.append({
                "violation_tag": clean_text(t).upper(),
                "id": row["id"],
                "vehicle_number": row["vehicle_number"],
                "hotspot_unit": row["hotspot_unit"],
            })

tag_df = pd.DataFrame(tag_rows)

violation_tag_summary = (
    tag_df.groupby("violation_tag")
    .agg(
        tag_frequency=("id", "size"),
        unique_vehicles=("vehicle_number", "nunique"),
        unique_hotspots=("hotspot_unit", "nunique"),
    )
    .reset_index()
    .sort_values("tag_frequency", ascending=False)
)

severity_distribution = (
    df.groupby("severity_score")
    .size()
    .reset_index(name="count")
    .sort_values("severity_score")
)

violation_by_vehicle_type = (
    df.groupby("vehicle_type")
    .agg(
        records=("id", "size"),
        mean_severity=("severity_score", "mean"),
        approval_rate=("validation_status_clean", lambda s: (s == "approved").mean()),
    )
    .reset_index()
    .sort_values("records", ascending=False)
)

violation_by_hotspot = (
    df.groupby("hotspot_unit")
    .agg(
        records=("id", "size"),
        mean_severity=("severity_score", "mean"),
        approval_rate=("validation_status_clean", lambda s: (s == "approved").mean()),
        unique_vehicles=("vehicle_number", "nunique"),
    )
    .reset_index()
    .sort_values(["records", "mean_severity"], ascending=[False, False])
)

# Tag co-occurrence patterns
combo_counts = (
    df["violation_tags"]
    .apply(lambda x: tuple(sorted([clean_text(t).upper() for t in x])) if x else ("UNKNOWN",))
    .value_counts()
    .reset_index()
)
combo_counts.columns = ["violation_combo", "frequency"]
combo_counts["violation_combo"] = combo_counts["violation_combo"].apply(lambda x: " | ".join(x))

# -------------------------
# Consolidated Phase 1 feature table
# -------------------------
phase1_features = df.copy()

# Useful derived columns for later stages
phase1_features["validation_is_approved"] = (phase1_features["validation_status_clean"] == "approved").astype(int)
phase1_features["validation_is_rejected"] = (phase1_features["validation_status_clean"] == "rejected").astype(int)
phase1_features["vehicle_total_violations"] = phase1_features.groupby("vehicle_number")["id"].transform("count")
phase1_features["vehicle_chronic_flag"] = (phase1_features["vehicle_total_violations"] >= 5).astype(int)
phase1_features["vehicle_repeat_flag"] = (phase1_features["vehicle_total_violations"] >= 2).astype(int)

phase1_features["hotspot_total_records"] = phase1_features.groupby("hotspot_unit")["id"].transform("count")
phase1_features["hotspot_approval_rate"] = phase1_features.groupby("hotspot_unit")["validation_is_approved"].transform("mean")
phase1_features["hotspot_uncertainty"] = 1.0 - phase1_features["hotspot_approval_rate"]

phase1_features["vehicle_type_total_records"] = phase1_features.groupby("vehicle_type")["id"].transform("count")
phase1_features["vehicle_type_mean_severity"] = phase1_features.groupby("vehicle_type")["severity_score"].transform("mean")

phase1_features["tag_frequency_record"] = phase1_features["dominant_violation_tag"].map(
    violation_tag_summary.set_index("violation_tag")["tag_frequency"]
).fillna(0).astype(int)

# -------------------------
# Save outputs
# -------------------------
validation_summary.to_csv(OUT_DIR / "validation_summary.csv", index=False)
validation_by_vehicle_type.to_csv(OUT_DIR / "validation_by_vehicle_type.csv", index=False)
validation_by_police_station.to_csv(OUT_DIR / "validation_by_police_station.csv", index=False)
validation_by_hotspot.to_csv(OUT_DIR / "validation_by_hotspot.csv", index=False)

vehicle_summary.to_csv(OUT_DIR / "vehicle_summary.csv", index=False)
top_vehicle_offenders.to_csv(OUT_DIR / "top_vehicle_offenders.csv", index=False)
chronic_offenders.to_csv(OUT_DIR / "chronic_offenders.csv", index=False)
vehicle_type_summary.to_csv(OUT_DIR / "vehicle_type_summary.csv", index=False)

hourly_distribution.to_csv(OUT_DIR / "hourly_distribution.csv", index=False)
daily_distribution.to_csv(OUT_DIR / "daily_distribution.csv", index=False)
weekly_distribution.to_csv(OUT_DIR / "weekly_distribution.csv", index=False)
monthly_distribution.to_csv(OUT_DIR / "monthly_distribution.csv", index=False)
temporal_by_hour_vehicle_type.to_csv(OUT_DIR / "temporal_by_hour_vehicle_type.csv", index=False)
temporal_by_day_validation.to_csv(OUT_DIR / "temporal_by_day_validation.csv", index=False)
weekly_trend_by_hotspot.to_csv(OUT_DIR / "weekly_trend_by_hotspot.csv", index=False)

violation_tag_summary.to_csv(OUT_DIR / "violation_tag_summary.csv", index=False)
severity_distribution.to_csv(OUT_DIR / "severity_distribution.csv", index=False)
violation_by_vehicle_type.to_csv(OUT_DIR / "violation_by_vehicle_type.csv", index=False)
violation_by_hotspot.to_csv(OUT_DIR / "violation_by_hotspot.csv", index=False)
combo_counts.to_csv(OUT_DIR / "violation_combo_summary.csv", index=False)

phase1_features.to_csv(OUT_DIR / "phase1_enriched_dataset.csv", index=False)

# -------------------------
# Print summary
# -------------------------
print("Phase 1 complete")
print("Total rows:", len(df))
print("Approved rows:", int((df["validation_status_clean"] == "approved").sum()))
print("Approval rate:", round(approval_rate * 100, 2), "%")
print("Unique vehicles:", df["vehicle_number"].nunique())
print("Unique hotspots:", df["hotspot_unit"].nunique())
print("Peak hour:", peak_hour)
print("Peak day:", peak_day)
print("Chronic offenders (>=5):", len(chronic_offenders))
print("Outputs saved to:", OUT_DIR.resolve())

# -------------------------
# Plots
# -------------------------
plt.figure(figsize=(10, 5))
plt.bar(validation_summary["validation_status"].astype(str), validation_summary["count"])
plt.title("Validation Status Distribution")
plt.xlabel("Validation Status")
plt.ylabel("Count")
save_plot(OUT_DIR / "validation_status_distribution.png")

plt.figure(figsize=(12, 5))
plt.bar(hourly_distribution["hour_ist"].astype(int), hourly_distribution["count"])
plt.title("Hourly Violation Distribution (IST)")
plt.xlabel("Hour")
plt.ylabel("Count")
save_plot(OUT_DIR / "hourly_distribution.png")

plt.figure(figsize=(12, 5))
plt.bar(daily_distribution["day_of_week"].astype(str), daily_distribution["count"])
plt.title("Day-of-Week Violation Distribution")
plt.xlabel("Day")
plt.ylabel("Count")
plt.xticks(rotation=30)
save_plot(OUT_DIR / "daily_distribution.png")

plt.figure(figsize=(12, 6))
top_tags = violation_tag_summary.head(TOP_N)
plt.barh(top_tags["violation_tag"][::-1], top_tags["tag_frequency"][::-1])
plt.title("Top Violation Tags")
plt.xlabel("Frequency")
plt.ylabel("Violation Tag")
save_plot(OUT_DIR / "top_violation_tags.png")

plt.figure(figsize=(12, 6))
top_vehicles = top_vehicle_offenders.head(TOP_N).sort_values("total_violations")
plt.barh(top_vehicles["vehicle_number"], top_vehicles["total_violations"])
plt.title("Top Vehicle Offenders")
plt.xlabel("Total Violations")
plt.ylabel("Vehicle Number")
save_plot(OUT_DIR / "top_vehicle_offenders.png")

Phase 1 complete
Total rows: 298450
Approved rows: 115400
Approval rate: 38.67 %
Unique vehicles: 231890
Unique hotspots: 222
Peak hour: 10
Peak day: Sunday
Chronic offenders (>=5): 3489
Outputs saved to: /content/phase1_outputs_1


In [ ]:
import ast
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

INPUT_CSV = "jan to may police violation_anonymized791b166.csv"
PHASE1_PATH = Path("phase1_outputs_1/phase1_enriched_dataset.csv")
OUT_DIR = Path("phase2_outputs_1")
OUT_DIR.mkdir(parents=True, exist_ok=True)

EPS = 1e-9
TOP_N = 25

SEVERITY_RULES = {
    5: [
        "DOUBLE PARKING",
        "NEAR ROAD CROSSING",
        "NEAR TRAFFIC LIGHT",
        "NEAR ZEBRA CROSSING",
        "NEAR TRAFFIC LIGHT / ZEBRA CROSSING",
        "NEAR TRAFFIC LIGHT/ZEBRA CROSSING",
    ],
    4: [
        "PARKING IN MAIN ROAD",
        "NEAR BUS STOP",
        "NEAR SCHOOL",
        "NEAR HOSPITAL",
        "OPPOSITE ANOTHER VEHICLE",
    ],
    3: [
        "PARKING ON FOOTPATH",
    ],
    2: [
        "WRONG PARKING",
        "PARKING OTHER THAN BUS STOP",
    ],
    1: [
        "NO PARKING",
        "NO PARKING (GENERIC)",
    ],
}

def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def parse_listlike(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    s = str(value).strip()
    if not s:
        return []
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, (list, tuple)):
            return list(parsed)
        return [parsed]
    except Exception:
        if s.startswith("[") and s.endswith("]"):
            s = s[1:-1]
        parts = [p.strip().strip("'").strip('"') for p in s.split(",")]
        return [p for p in parts if p]

def normalize_tag(tag):
    s = clean_text(tag).upper()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"\s*/\s*", " / ", s)
    s = s.replace("&", "AND")
    return s.strip()

def make_hotspot_unit(row):
    junction = clean_text(row.get("junction_name", ""))
    if junction and junction.upper() != "NO JUNCTION":
        return f"JUNCTION::{junction}"
    station = clean_text(row.get("police_station", "UNKNOWN"))
    if not station:
        station = "UNKNOWN"
    return f"POLICE_STATION::{station}"

def severity_for_tags(tags):
    if not tags:
        return 1, [], "NO PARKING"

    normalized = [normalize_tag(t) for t in tags]
    hit_tags = []
    best = 1

    for tag in normalized:
        tag_best = 1
        matched = False
        for sev in sorted(SEVERITY_RULES.keys(), reverse=True):
            for pattern in SEVERITY_RULES[sev]:
                p = normalize_tag(pattern)
                if p == tag or p in tag:
                    tag_best = sev
                    matched = True
                    break
            if matched:
                break
        best = max(best, tag_best)
        if matched:
            hit_tags.append(tag)

    dominant = hit_tags[0] if hit_tags else normalized[0]
    return best, hit_tags, dominant

def safe_ratio(num, den):
    return num / (den + EPS)

def save_plot(path):
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()

def main():
    source_path = PHASE1_PATH if PHASE1_PATH.exists() else Path(INPUT_CSV)
    df = pd.read_csv(source_path, low_memory=False)

    if "violation_type" not in df.columns:
        raise ValueError("Missing required column: violation_type")

    if "hotspot_unit" not in df.columns:
        if {"junction_name", "police_station"}.issubset(df.columns):
            df["hotspot_unit"] = df.apply(make_hotspot_unit, axis=1)
        else:
            df["hotspot_unit"] = "UNKNOWN"

    if "validation_status" in df.columns:
        df["validation_status_clean"] = df["validation_status"].fillna("").astype(str).str.lower()
    else:
        df["validation_status_clean"] = "approved"

    df["violation_tags"] = df["violation_type"].apply(parse_listlike)
    df["violation_tag_count"] = df["violation_tags"].apply(len)

    sev_res = df["violation_tags"].apply(severity_for_tags)
    df["severity_score"] = sev_res.apply(lambda x: int(x[0]))
    df["matched_severity_tags"] = sev_res.apply(lambda x: x[1])
    df["dominant_violation_tag"] = sev_res.apply(lambda x: x[2])
    df["severity_normalized"] = df["severity_score"] / 5.0

    df["severity_is_very_high"] = (df["severity_score"] == 5).astype(int)
    df["severity_is_high"] = (df["severity_score"] == 4).astype(int)
    df["severity_is_moderate"] = (df["severity_score"] == 3).astype(int)
    df["severity_is_low"] = (df["severity_score"] == 2).astype(int)
    df["severity_is_baseline"] = (df["severity_score"] == 1).astype(int)

    approved_mask = df["validation_status_clean"].eq("approved")
    approved = df.loc[approved_mask].copy()

    severity_record_cols = [
        c for c in [
            "id", "created_datetime", "created_datetime_parsed", "created_datetime_ist",
            "latitude", "longitude", "vehicle_number", "vehicle_type",
            "police_station", "junction_name", "hotspot_unit",
            "validation_status", "validation_status_clean",
            "violation_type", "violation_tags", "violation_tag_count",
            "severity_score", "severity_normalized", "matched_severity_tags",
            "dominant_violation_tag"
        ] if c in df.columns
    ]

    hotspot_summary = (
        approved.groupby("hotspot_unit", dropna=False)
        .agg(
            approved_records=("severity_score", "size"),
            mean_severity=("severity_score", "mean"),
            median_severity=("severity_score", "median"),
            severity_sum=("severity_score", "sum"),
            very_high_count=("severity_is_very_high", "sum"),
            high_count=("severity_is_high", "sum"),
            moderate_count=("severity_is_moderate", "sum"),
            low_count=("severity_is_low", "sum"),
            baseline_count=("severity_is_baseline", "sum"),
            unique_vehicles=("vehicle_number", "nunique") if "vehicle_number" in approved.columns else ("severity_score", "size"),
        )
        .reset_index()
    )

    hotspot_summary["severity_share"] = safe_ratio(hotspot_summary["severity_sum"], hotspot_summary["severity_sum"].sum())
    hotspot_summary["severity_priority_score"] = (
        0.50 * (hotspot_summary["severity_sum"] / (hotspot_summary["severity_sum"].max() + EPS)) +
        0.30 * (hotspot_summary["mean_severity"] / 5.0) +
        0.20 * safe_ratio(hotspot_summary["approved_records"], hotspot_summary["approved_records"].max())
    ) * 100.0

    hotspot_summary = hotspot_summary.sort_values(
        ["severity_priority_score", "severity_sum", "approved_records"],
        ascending=[False, False, False],
    ).reset_index(drop=True)
    hotspot_summary["severity_rank"] = np.arange(1, len(hotspot_summary) + 1)

    vehicle_type_summary = (
        approved.groupby("vehicle_type", dropna=False)
        .agg(
            approved_records=("severity_score", "size"),
            mean_severity=("severity_score", "mean"),
            median_severity=("severity_score", "median"),
            severity_sum=("severity_score", "sum"),
            unique_vehicles=("vehicle_number", "nunique") if "vehicle_number" in approved.columns else ("severity_score", "size"),
        )
        .reset_index()
        .sort_values(["severity_sum", "approved_records"], ascending=[False, False])
    )

    tag_rows = []
    for _, row in approved.iterrows():
        tags = row["violation_tags"]
        if not tags:
            tag_rows.append({"violation_tag": "NO TAG", "severity_score": row["severity_score"], "hotspot_unit": row["hotspot_unit"]})
        else:
            for t in tags:
                tag_rows.append({
                    "violation_tag": normalize_tag(t),
                    "severity_score": row["severity_score"],
                    "hotspot_unit": row["hotspot_unit"],
                })

    tag_df = pd.DataFrame(tag_rows)
    if tag_df.empty:
        tag_summary = pd.DataFrame(columns=["violation_tag", "frequency", "avg_record_severity", "hotspot_count"])
    else:
        tag_summary = (
            tag_df.groupby("violation_tag", dropna=False)
            .agg(
                frequency=("violation_tag", "size"),
                avg_record_severity=("severity_score", "mean"),
                hotspot_count=("hotspot_unit", "nunique"),
            )
            .reset_index()
            .sort_values(["frequency", "avg_record_severity"], ascending=[False, False])
        )

    validation_severity = (
        df.groupby("validation_status_clean", dropna=False)
        .agg(
            records=("severity_score", "size"),
            mean_severity=("severity_score", "mean"),
            severity_sum=("severity_score", "sum"),
        )
        .reset_index()
        .sort_values("records", ascending=False)
    )

    df[severity_record_cols].to_csv(OUT_DIR / "phase2_enriched_dataset.csv", index=False)
    approved[severity_record_cols].to_csv(OUT_DIR / "phase2_approved_severity_dataset.csv", index=False)
    hotspot_summary.to_csv(OUT_DIR / "phase2_hotspot_severity_scores.csv", index=False)
    vehicle_type_summary.to_csv(OUT_DIR / "phase2_vehicle_type_severity.csv", index=False)
    tag_summary.to_csv(OUT_DIR / "phase2_violation_tag_severity.csv", index=False)
    validation_severity.to_csv(OUT_DIR / "phase2_validation_severity.csv", index=False)

    print("Phase 2 complete")
    print("Input source:", source_path)
    print("Total rows:", len(df))
    print("Approved rows:", len(approved))
    print("Hotspots scored:", len(hotspot_summary))
    print("Mean severity (approved):", round(float(approved["severity_score"].mean()) if len(approved) else 0.0, 4))
    print("Top hotspot:", hotspot_summary.iloc[0]["hotspot_unit"] if len(hotspot_summary) else "N/A")
    print("Outputs saved to:", OUT_DIR.resolve())

    if len(hotspot_summary):
        plt.figure(figsize=(12, 6))
        top_hotspots = hotspot_summary.head(TOP_N).sort_values("severity_priority_score", ascending=True)
        plt.barh(top_hotspots["hotspot_unit"], top_hotspots["severity_priority_score"])
        plt.title("Top Hotspots by Severity Priority Score")
        plt.xlabel("Severity Priority Score")
        plt.ylabel("Hotspot")
        save_plot(OUT_DIR / "top_hotspots_by_severity_priority.png")

    if len(tag_summary):
        plt.figure(figsize=(12, 6))
        top_tags = tag_summary.head(TOP_N).sort_values("frequency", ascending=True)
        plt.barh(top_tags["violation_tag"], top_tags["frequency"])
        plt.title("Top Violation Tags (Approved Records)")
        plt.xlabel("Frequency")
        plt.ylabel("Violation Tag")
        save_plot(OUT_DIR / "top_violation_tags_phase2.png")

    if len(validation_severity):
        plt.figure(figsize=(10, 5))
        plt.bar(validation_severity["validation_status_clean"].astype(str), validation_severity["records"])
        plt.title("Records by Validation Status")
        plt.xlabel("Validation Status")
        plt.ylabel("Records")
        plt.xticks(rotation=30)
        save_plot(OUT_DIR / "validation_status_phase2.png")

if __name__ == "__main__":
    main()

Phase 2 complete
Input source: phase1_outputs_1/phase1_enriched_dataset.csv
Total rows: 298450
Approved rows: 115400
Hotspots scored: 220
Mean severity (approved): 1.5732
Top hotspot: POLICE_STATION::HAL Old Airport
Outputs saved to: /content/phase2_outputs_1


In [ ]:
# =========================
# Phase 3 — ST-DBSCAN Spatial + Temporal Hotspot Discovery
# Input  : Phase 2 approved, severity-scored dataset (preferred)
# Output : Clustered records, cluster summary, noise points, weekly cluster counts
# =========================

import ast
import warnings
from collections import deque
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.neighbors import BallTree

warnings.filterwarnings("ignore")

# -------------------------
# Config
# -------------------------
RAW_CSV = "jan to may police violation_anonymized791b166.csv"
PHASE2_APPROVED_PATH = Path("phase2_outputs_1/phase2_approved_severity_dataset.csv")
PHASE2_ENRICHED_PATH = Path("phase2_outputs_1/phase2_enriched_dataset.csv")

OUT_DIR = Path("phase3_outputs_1")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ST-DBSCAN parameters from the concept note
EPS_SPATIAL_METERS = 150.0
EPS_TEMPORAL_DAYS = 3.0
MIN_PTS = 15

EPS = 1e-9
TOP_N = 25

# -------------------------
# Helpers
# -------------------------
def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def parse_listlike(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    s = str(value).strip()
    if not s:
        return []
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, (list, tuple)):
            return list(parsed)
        return [parsed]
    except Exception:
        if s.startswith("[") and s.endswith("]"):
            s = s[1:-1]
        parts = [p.strip().strip("'").strip('"') for p in s.split(",")]
        return [p for p in parts if p]

def normalize_tag(tag):
    return clean_text(tag).upper().replace("&", "AND").strip()

def severity_from_tags(tags):
    """
    Fallback severity mapping if Phase 2 output is unavailable.
    """
    severity_map = {
        5: {
            "DOUBLE PARKING",
            "NEAR ROAD CROSSING",
            "NEAR TRAFFIC LIGHT",
            "NEAR ZEBRA CROSSING",
            "NEAR TRAFFIC LIGHT / ZEBRA CROSSING",
            "NEAR TRAFFIC LIGHT/ZEBRA CROSSING",
        },
        4: {
            "PARKING IN MAIN ROAD",
            "NEAR BUS STOP",
            "NEAR SCHOOL",
            "NEAR HOSPITAL",
            "OPPOSITE ANOTHER VEHICLE",
        },
        3: {"PARKING ON FOOTPATH"},
        2: {"WRONG PARKING", "PARKING OTHER THAN BUS STOP"},
        1: {"NO PARKING", "NO PARKING (GENERIC)"},
    }
    if not tags:
        return 1
    normalized = [normalize_tag(t) for t in tags]
    score = 1
    for sev in sorted(severity_map.keys(), reverse=True):
        vocab = severity_map[sev]
        if any(any(v == tag or v in tag for v in vocab) for tag in normalized):
            score = sev
            break
    return score

def week_start_monday(series):
    dt = pd.to_datetime(series, errors="coerce", utc=True).dt.tz_convert("Asia/Kolkata")
    week_start = dt.dt.normalize() - pd.to_timedelta(dt.dt.weekday, unit="D")
    return week_start.dt.tz_localize(None)

def mode_or_default(s, default=""):
    s = pd.Series(s).dropna().astype(str)
    if s.empty:
        return default
    m = s.mode()
    if m.empty:
        return default
    return m.iloc[0]

def make_hotspot_unit(row):
    j = clean_text(row.get("junction_name", ""))
    if j and j.upper() != "NO JUNCTION":
        return f"JUNCTION::{j}"
    ps = clean_text(row.get("police_station", "UNKNOWN"))
    if not ps:
        ps = "UNKNOWN"
    return f"POLICE_STATION::{ps}"

def safe_ratio(a, b):
    return a / (b + EPS)

def save_plot(path):
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()

# -------------------------
# Load input
# -------------------------
def load_input():
    if PHASE2_APPROVED_PATH.exists():
        df = pd.read_csv(PHASE2_APPROVED_PATH, low_memory=False)
    elif PHASE2_ENRICHED_PATH.exists():
        df = pd.read_csv(PHASE2_ENRICHED_PATH, low_memory=False)
    else:
        df = pd.read_csv(RAW_CSV, low_memory=False)
    return df

# -------------------------
# ST-DBSCAN core
# -------------------------
def build_spatial_coordinates(lat, lon):
    """
    Local tangent-plane approximation in meters.
    Good enough for a city-scale epsilon of 150 m.
    """
    lat = np.asarray(lat, dtype=float)
    lon = np.asarray(lon, dtype=float)

    lat0 = np.deg2rad(np.nanmean(lat))
    lon0 = np.deg2rad(np.nanmean(lon))
    R = 6371000.0

    lat_rad = np.deg2rad(lat)
    lon_rad = np.deg2rad(lon)

    x = R * (lon_rad - lon0) * np.cos(lat0)
    y = R * (lat_rad - lat0)
    return np.c_[x, y]

def st_dbscan(coords_xy, time_days, eps_spatial_m, eps_temporal_days, min_pts):
    """
    ST-DBSCAN implementation using:
    - spatial radius search via BallTree in meters
    - temporal filtering in days
    - BFS expansion exactly like DBSCAN-style density reachability
    """
    n = len(coords_xy)
    tree = BallTree(coords_xy, metric="euclidean")

    labels = np.full(n, -99, dtype=int)   # -99 = unassigned, -1 = noise
    visited = np.zeros(n, dtype=bool)
    core_mask = np.zeros(n, dtype=bool)
    neighbor_cache = {}

    def region_query(i):
        if i in neighbor_cache:
            return neighbor_cache[i]
        idx = tree.query_radius(coords_xy[i:i+1], r=eps_spatial_m, return_distance=False)[0]
        if idx.size == 0:
            neighbor_cache[i] = idx
            return idx
        temporal_ok = np.abs(time_days[idx] - time_days[i]) <= eps_temporal_days
        neigh = idx[temporal_ok]
        neighbor_cache[i] = neigh
        return neigh

    cluster_id = 0

    for i in range(n):
        if visited[i]:
            continue

        visited[i] = True
        neighbors = region_query(i)

        if len(neighbors) < min_pts:
            labels[i] = -1
            continue

        cluster_id += 1
        labels[i] = cluster_id
        core_mask[i] = True

        seed_queue = deque()
        queued = set()

        for j in neighbors:
            if j != i and j not in queued:
                seed_queue.append(j)
                queued.add(j)

        while seed_queue:
            j = seed_queue.popleft()

            if not visited[j]:
                visited[j] = True
                j_neighbors = region_query(j)
                if len(j_neighbors) >= min_pts:
                    core_mask[j] = True
                    for k in j_neighbors:
                        if k not in queued:
                            seed_queue.append(k)
                            queued.add(k)

            if labels[j] in (-99, -1):
                labels[j] = cluster_id

    return labels, core_mask, neighbor_cache

# -------------------------
# Main
# -------------------------
def main():
    df = load_input()

    required = {
        "id", "latitude", "longitude", "vehicle_number", "vehicle_type",
        "violation_type", "created_datetime", "police_station", "junction_name"
    }
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    # Ensure time columns
    if "created_datetime_ist" not in df.columns:
        df["created_datetime_parsed"] = pd.to_datetime(df["created_datetime"], errors="coerce", utc=True)
        df["created_datetime_ist"] = df["created_datetime_parsed"].dt.tz_convert("Asia/Kolkata")
    else:
        df["created_datetime_ist"] = pd.to_datetime(df["created_datetime_ist"], errors="coerce")
        if getattr(df["created_datetime_ist"].dt, "tz", None) is None:
            # If the column is naive, treat it as IST
            df["created_datetime_ist"] = df["created_datetime_ist"].dt.tz_localize("Asia/Kolkata")

    df["created_datetime_ist_naive"] = df["created_datetime_ist"].dt.tz_localize(None)
    df["week_start"] = week_start_monday(df["created_datetime"])

    # If Phase 2 output is not available, create the severity score here
    if "severity_score" not in df.columns:
        df["violation_tags"] = df["violation_type"].apply(parse_listlike)
        df["severity_score"] = df["violation_tags"].apply(severity_from_tags)

    if "validation_status_clean" not in df.columns:
        if "validation_status" in df.columns:
            df["validation_status_clean"] = df["validation_status"].fillna("").astype(str).str.lower()
        else:
            df["validation_status_clean"] = "approved"

    # Stage 3 uses the approved evidence base
    approved = df.loc[df["validation_status_clean"].eq("approved")].copy()
    approved = approved.dropna(subset=["latitude", "longitude", "created_datetime_ist_naive"]).copy()
    approved = approved.reset_index(drop=True)

    if approved.empty:
        raise ValueError("No approved rows with valid latitude/longitude/timestamp were found.")

    # Hotspot unit fallback if not present
    if "hotspot_unit" not in approved.columns:
        approved["hotspot_unit"] = approved.apply(make_hotspot_unit, axis=1)

    # Spatial + temporal numeric arrays
    coords_xy = build_spatial_coordinates(approved["latitude"].values, approved["longitude"].values)
    time_days = (
        approved["created_datetime_ist_naive"].astype("int64").values.astype(np.float64)
        / 86_400_000_000_000.0
    )

    print("Approved rows:", len(approved))
    print("Unique hotspots (pre-clustering):", approved["hotspot_unit"].nunique())
    print("Running ST-DBSCAN with:")
    print(f"  spatial epsilon = {EPS_SPATIAL_METERS:.1f} meters")
    print(f"  temporal epsilon = {EPS_TEMPORAL_DAYS:.1f} days")
    print(f"  MinPts          = {MIN_PTS}")

    labels, core_mask, neighbor_cache = st_dbscan(
        coords_xy=coords_xy,
        time_days=time_days,
        eps_spatial_m=EPS_SPATIAL_METERS,
        eps_temporal_days=EPS_TEMPORAL_DAYS,
        min_pts=MIN_PTS,
    )

    approved["st_dbscan_cluster_id"] = labels
    approved["is_core_point"] = core_mask.astype(int)
    approved["is_noise"] = (approved["st_dbscan_cluster_id"] == -1).astype(int)

    # Human-readable cluster labels
    def cluster_label_for_group(g):
        named_junctions = g["junction_name"].fillna("").astype(str)
        named_junctions = named_junctions[named_junctions.str.upper().ne("NO JUNCTION")]
        named_junctions = named_junctions[named_junctions.str.strip().ne("")]
        if len(named_junctions) > 0:
            return mode_or_default(named_junctions, default="")
        return ""

    cluster_summary = []
    cluster_ids = sorted([c for c in approved["st_dbscan_cluster_id"].unique() if c != -1])

    for cid in cluster_ids:
        g = approved[approved["st_dbscan_cluster_id"] == cid].copy()

        x = coords_xy[g.index.values, 0]
        y = coords_xy[g.index.values, 1]

        # Approximate spread and centroid
        centroid_x = float(np.mean(x))
        centroid_y = float(np.mean(y))
        spatial_radius_m = float(np.max(np.sqrt((x - centroid_x) ** 2 + (y - centroid_y) ** 2))) if len(g) else 0.0

        cluster_name = cluster_label_for_group(g)
        if not cluster_name:
            # Fall back to police station, then coordinates
            station_name = mode_or_default(g["police_station"], default="")
            if station_name and station_name.upper() != "NO JUNCTION":
                cluster_name = f"{station_name}"
            else:
                cluster_name = f"Cluster {cid}"

        cluster_summary.append({
            "st_dbscan_cluster_id": cid,
            "cluster_label": cluster_name,
            "records": len(g),
            "core_points": int(g["is_core_point"].sum()),
            "noise_points": int(g["is_noise"].sum()),
            "start_time_ist": g["created_datetime_ist"].min(),
            "end_time_ist": g["created_datetime_ist"].max(),
            "duration_days": float((g["created_datetime_ist"].max() - g["created_datetime_ist"].min()).total_seconds() / 86400.0) if len(g) else 0.0,
            "latitude_mean": float(g["latitude"].mean()),
            "longitude_mean": float(g["longitude"].mean()),
            "severity_mean": float(g["severity_score"].mean()) if "severity_score" in g.columns else np.nan,
            "severity_sum": float(g["severity_score"].sum()) if "severity_score" in g.columns else np.nan,
            "unique_vehicles": int(g["vehicle_number"].nunique()) if "vehicle_number" in g.columns else 0,
            "dominant_vehicle_type": mode_or_default(g["vehicle_type"], default=""),
            "dominant_police_station": mode_or_default(g["police_station"], default=""),
            "dominant_junction_name": mode_or_default(g["junction_name"], default=""),
            "dominant_violation_tag": mode_or_default(
                g["violation_type"].apply(lambda x: normalize_tag(parse_listlike(x)[0]) if parse_listlike(x) else "NO TAG"),
                default="NO TAG",
            ),
            "approx_spatial_radius_m": spatial_radius_m,
        })

    cluster_summary_df = pd.DataFrame(cluster_summary).sort_values(
        ["records", "severity_sum"],
        ascending=[False, False]
    ).reset_index(drop=True)

    # Trend-friendly weekly cluster counts for later phase
    approved["cluster_week_start"] = approved["created_datetime_ist"].dt.to_period("W-MON").dt.start_time
    weekly_cluster_counts = (
        approved[approved["st_dbscan_cluster_id"] != -1]
        .groupby(["st_dbscan_cluster_id", "cluster_week_start"])
        .size()
        .reset_index(name="weekly_count")
        .sort_values(["st_dbscan_cluster_id", "cluster_week_start"])
    )

    # Merge cluster label back to records
    approved = approved.merge(
        cluster_summary_df[["st_dbscan_cluster_id", "cluster_label"]],
        on="st_dbscan_cluster_id",
        how="left"
    )
    approved["cluster_label"] = approved["cluster_label"].fillna("Noise")

    # Noise rows
    noise_df = approved[approved["st_dbscan_cluster_id"] == -1].copy()

    # Output files
    approved.to_csv(OUT_DIR / "phase3_clustered_dataset.csv", index=False)
    cluster_summary_df.to_csv(OUT_DIR / "phase3_cluster_summary.csv", index=False)
    weekly_cluster_counts.to_csv(OUT_DIR / "phase3_cluster_weekly_counts.csv", index=False)
    noise_df.to_csv(OUT_DIR / "phase3_noise_points.csv", index=False)

    # Helpful compact dispatch-style table for later stages
    dispatch_view = cluster_summary_df[[
        "st_dbscan_cluster_id", "cluster_label", "records", "severity_sum", "severity_mean",
        "unique_vehicles", "dominant_vehicle_type", "dominant_police_station",
        "dominant_junction_name", "approx_spatial_radius_m", "start_time_ist", "end_time_ist"
    ]].copy()
    dispatch_view.to_csv(OUT_DIR / "phase3_cluster_dispatch_view.csv", index=False)

    # Summary printout
    n_clusters = len(cluster_summary_df)
    n_noise = int((approved["st_dbscan_cluster_id"] == -1).sum())
    print("\nPhase 3 complete")
    print("Clusters found:", n_clusters)
    print("Noise points   :", n_noise)
    print("Clustered rows  :", len(approved) - n_noise)
    print("Output folder   :", OUT_DIR.resolve())

    if n_clusters > 0:
        print("\nTop 10 clusters:")
        print(
            cluster_summary_df[[
                "st_dbscan_cluster_id", "cluster_label", "records",
                "severity_sum", "severity_mean", "unique_vehicles"
            ]].head(10).to_string(index=False)
        )

    # Plots
    if len(cluster_summary_df):
        plt.figure(figsize=(12, 6))
        top_clusters = cluster_summary_df.head(TOP_N).sort_values("records", ascending=True)
        plt.barh(top_clusters["cluster_label"], top_clusters["records"])
        plt.title("Top ST-DBSCAN Clusters by Record Count")
        plt.xlabel("Records")
        plt.ylabel("Cluster")
        save_plot(OUT_DIR / "top_clusters_by_records.png")

        plt.figure(figsize=(12, 6))
        top_sev = cluster_summary_df.head(TOP_N).sort_values("severity_sum", ascending=True)
        plt.barh(top_sev["cluster_label"], top_sev["severity_sum"])
        plt.title("Top ST-DBSCAN Clusters by Severity Sum")
        plt.xlabel("Severity Sum")
        plt.ylabel("Cluster")
        save_plot(OUT_DIR / "top_clusters_by_severity.png")

        plt.figure(figsize=(10, 5))
        plt.hist(cluster_summary_df["records"], bins=30)
        plt.title("Cluster Size Distribution")
        plt.xlabel("Records per Cluster")
        plt.ylabel("Frequency")
        save_plot(OUT_DIR / "cluster_size_distribution.png")

if __name__ == "__main__":
    main()

Approved rows: 115400
Unique hotspots (pre-clustering): 220
Running ST-DBSCAN with:
  spatial epsilon = 150.0 meters
  temporal epsilon = 3.0 days
  MinPts          = 15

Phase 3 complete
Clusters found: 798
Noise points   : 23068
Clustered rows  : 92332
Output folder   : /content/phase3_outputs_1

Top 10 clusters:
 st_dbscan_cluster_id                          cluster_label  records  severity_sum  severity_mean  unique_vehicles
                    2                BTP040 - Elite Junction    21733       34509.0       1.587862            18627
                    3         BTP051 - Safina Plaza Junction     9287       15028.0       1.618176             7622
                   32          BTP027 - Modi Bridge Junction     3168        5271.0       1.663826             2725
                   19 BTP001 - 10th Cross, Dr. Rajkumar Road     2934        4795.0       1.634288             2620
                   35       BTP020 - Hosahalli Metro Station     1650        2485.0       1.506061     

In [ ]:
# =========================
# Phase 4 — Implicit μ Estimation / Dwell-Time Estimation
# Input  : Phase 3 clustered approved records
# Output : Gap-level dwell events, cluster-level μ summary, vehicle-type μ summary,
#          merged dataset for Stage 5
# =========================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# -------------------------
# Config
# -------------------------
PHASE3_PATHS = [
    Path("phase3_outputs_1/phase3_clustered_dataset.csv"),
    Path("phase3_outputs_1/phase3_cluster_dispatch_view.csv"),
]

OUT_DIR = Path("phase4_outputs_1")
OUT_DIR.mkdir(parents=True, exist_ok=True)

EPS = 1e-9
TOP_N = 25

# -------------------------
# Helpers
# -------------------------
def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def safe_ratio(a, b):
    return a / (b + EPS)

def parse_datetime_to_ist(df: pd.DataFrame) -> pd.Series:
    """
    Prefer created_datetime_ist if present.
    Otherwise derive it from created_datetime, which is stored in UTC in the source file.
    Returns a timezone-aware IST series when possible.
    """
    if "created_datetime_ist" in df.columns:
        ts = pd.to_datetime(df["created_datetime_ist"], errors="coerce", utc=True)
        if ts.notna().any():
            return ts.dt.tz_convert("Asia/Kolkata")
        # fallback for naive strings
        ts = pd.to_datetime(df["created_datetime_ist"], errors="coerce")
        return ts.dt.tz_localize("Asia/Kolkata", nonexistent="NaT", ambiguous="NaT")

    if "created_datetime_parsed" in df.columns:
        ts = pd.to_datetime(df["created_datetime_parsed"], errors="coerce", utc=True)
        return ts.dt.tz_convert("Asia/Kolkata")

    if "created_datetime" not in df.columns:
        raise ValueError("Missing created_datetime / created_datetime_ist column.")

    ts = pd.to_datetime(df["created_datetime"], errors="coerce", utc=True)
    return ts.dt.tz_convert("Asia/Kolkata")

def get_cluster_col(df: pd.DataFrame) -> str:
    for c in ["st_dbscan_cluster_id", "cluster_id", "dbscan_cluster_id"]:
        if c in df.columns:
            return c
    raise ValueError("No cluster id column found. Expected st_dbscan_cluster_id from Phase 3.")

def get_vehicle_col(df: pd.DataFrame) -> str:
    """
    Document says vehicle_number. If updated_vehicle_number exists and has values,
    we use it as a canonical fallback because it is a corrected version from review.
    """
    if "updated_vehicle_number" in df.columns:
        # use updated_vehicle_number where present, otherwise vehicle_number
        if "vehicle_number" not in df.columns:
            df["vehicle_number"] = ""
        updated = df["updated_vehicle_number"].fillna("").astype(str).str.strip()
        original = df["vehicle_number"].fillna("").astype(str).str.strip()
        df["canonical_vehicle_number"] = np.where(updated.ne(""), updated, original)
        return "canonical_vehicle_number"

    if "vehicle_number" not in df.columns:
        raise ValueError("Missing vehicle_number / updated_vehicle_number column.")
    df["canonical_vehicle_number"] = df["vehicle_number"].fillna("").astype(str).str.strip()
    return "canonical_vehicle_number"

def get_vehicle_type_col(df: pd.DataFrame) -> str:
    if "updated_vehicle_type" in df.columns:
        if "vehicle_type" not in df.columns:
            df["vehicle_type"] = ""
        updated = df["updated_vehicle_type"].fillna("").astype(str).str.strip()
        original = df["vehicle_type"].fillna("").astype(str).str.strip()
        df["canonical_vehicle_type"] = np.where(updated.ne(""), updated, original)
        return "canonical_vehicle_type"

    if "vehicle_type" not in df.columns:
        df["canonical_vehicle_type"] = "UNKNOWN"
        return "canonical_vehicle_type"
    df["canonical_vehicle_type"] = df["vehicle_type"].fillna("UNKNOWN").astype(str).str.strip()
    df.loc[df["canonical_vehicle_type"].eq(""), "canonical_vehicle_type"] = "UNKNOWN"
    return "canonical_vehicle_type"

def make_hotspot_unit(row):
    junction = clean_text(row.get("junction_name", ""))
    if junction and junction.upper() != "NO JUNCTION":
        return f"JUNCTION::{junction}"
    station = clean_text(row.get("police_station", "UNKNOWN"))
    if not station:
        station = "UNKNOWN"
    return f"POLICE_STATION::{station}"

def save_plot(path):
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()

def load_phase3():
    for p in PHASE3_PATHS:
        if p.exists():
            return pd.read_csv(p, low_memory=False), p
    raise FileNotFoundError(
        "Phase 3 output not found. Expected phase3_outputs/phase3_clustered_dataset.csv"
    )

# -------------------------
# Main
# -------------------------
def main():
    df, src = load_phase3()
    print("Loaded:", src)

    # Required fields
    if "latitude" not in df.columns or "longitude" not in df.columns:
        raise ValueError("Phase 3 dataset must include latitude and longitude.")

    cluster_col = get_cluster_col(df)
    vehicle_col = get_vehicle_col(df)
    vehicle_type_col = get_vehicle_type_col(df)

    # Parse timestamps
    df["created_datetime_ist"] = parse_datetime_to_ist(df)
    df["created_datetime_ist_naive"] = df["created_datetime_ist"].dt.tz_localize(None)
    df["service_date"] = df["created_datetime_ist_naive"].dt.date

    # Keep approved clustered records only; exclude noise for dwell estimation
    if "validation_status_clean" in df.columns:
        df["validation_status_clean"] = df["validation_status_clean"].fillna("").astype(str).str.lower()
    elif "validation_status" in df.columns:
        df["validation_status_clean"] = df["validation_status"].fillna("").astype(str).str.lower()
    else:
        df["validation_status_clean"] = "approved"

    approved = df.loc[df["validation_status_clean"].eq("approved")].copy()
    approved = approved.dropna(subset=[cluster_col, vehicle_col, "created_datetime_ist_naive"]).copy()
    approved = approved.loc[approved[cluster_col].ne(-1)].copy()
    approved = approved.loc[approved[vehicle_col].fillna("").astype(str).str.strip().ne("")].copy()
    approved = approved.reset_index(drop=True)

    if approved.empty:
        raise ValueError("No approved clustered rows available for dwell-time estimation.")

    # If missing hotspot_unit, reconstruct it
    if "hotspot_unit" not in approved.columns:
        if {"junction_name", "police_station"}.issubset(approved.columns):
            approved["hotspot_unit"] = approved.apply(make_hotspot_unit, axis=1)
        else:
            approved["hotspot_unit"] = f"CLUSTER::{approved[cluster_col].astype(str)}"

    # Sort so groupby diff works correctly
    approved = approved.sort_values(
        [cluster_col, vehicle_col, "service_date", "created_datetime_ist_naive"]
    ).reset_index(drop=True)

    # Consecutive timestamp gaps within same vehicle, same cluster, same day
    group_cols = [cluster_col, vehicle_col, "service_date"]
    approved["prev_created_datetime_ist"] = approved.groupby(group_cols)["created_datetime_ist_naive"].shift(1)
    approved["gap_minutes"] = (
        approved["created_datetime_ist_naive"] - approved["prev_created_datetime_ist"]
    ).dt.total_seconds() / 60.0

    # Keep only valid positive gaps as lower-bound dwell-time estimates
    gaps = approved.loc[
        approved["gap_minutes"].notna() & (approved["gap_minutes"] > 0)
    ].copy()

    if gaps.empty:
        raise ValueError(
            "No valid inter-record gaps found. Check whether vehicles repeat within the same cluster and day."
        )

    # Output a record-level gap log
    gap_log_cols = [
        c for c in [
            "id", cluster_col, "hotspot_unit", vehicle_col, vehicle_type_col,
            "service_date", "prev_created_datetime_ist", "created_datetime_ist_naive",
            "gap_minutes", "latitude", "longitude", "junction_name", "police_station",
            "severity_score", "dominant_violation_tag"
        ] if c in gaps.columns
    ]
    gaps_out = gaps[gap_log_cols].copy()

    # -------------------------
    # Overall summary
    # -------------------------
    overall_mean_dwell = float(gaps_out["gap_minutes"].mean())
    overall_median_dwell = float(gaps_out["gap_minutes"].median())
    overall_mu = 60.0 / (overall_mean_dwell + EPS)

    overall_summary = pd.DataFrame([{
        "scope": "overall",
        "gap_count": int(len(gaps_out)),
        "unique_clusters_with_gaps": int(gaps_out[cluster_col].nunique()),
        "unique_vehicles_with_gaps": int(gaps_out[vehicle_col].nunique()),
        "mean_dwell_minutes": overall_mean_dwell,
        "median_dwell_minutes": overall_median_dwell,
        "mu_departures_per_hour": overall_mu,
    }])

    # -------------------------
    # Vehicle-type overall summary
    # -------------------------
    vehicle_type_summary = (
        gaps_out.groupby(vehicle_type_col, dropna=False)
        .agg(
            gap_count=("gap_minutes", "size"),
            unique_clusters=(cluster_col, "nunique"),
            unique_vehicles=(vehicle_col, "nunique"),
            mean_dwell_minutes=("gap_minutes", "mean"),
            median_dwell_minutes=("gap_minutes", "median"),
            std_dwell_minutes=("gap_minutes", lambda s: float(s.std(ddof=0)) if len(s) else 0.0),
        )
        .reset_index()
        .rename(columns={vehicle_type_col: "vehicle_type"})
        .sort_values(["gap_count", "mean_dwell_minutes"], ascending=[False, False])
    )
    vehicle_type_summary["mu_departures_per_hour"] = 60.0 / (vehicle_type_summary["mean_dwell_minutes"] + EPS)

    # -------------------------
    # Cluster-level overall summary
    # -------------------------
    cluster_overall_summary = (
        gaps_out.groupby(cluster_col, dropna=False)
        .agg(
            gap_count=("gap_minutes", "size"),
            unique_vehicles=(vehicle_col, "nunique"),
            unique_vehicle_types=(vehicle_type_col, "nunique"),
            mean_dwell_minutes=("gap_minutes", "mean"),
            median_dwell_minutes=("gap_minutes", "median"),
            std_dwell_minutes=("gap_minutes", lambda s: float(s.std(ddof=0)) if len(s) else 0.0),
            first_gap_time=("prev_created_datetime_ist", "min"),
            last_gap_time=("created_datetime_ist_naive", "max"),
        )
        .reset_index()
    )

    if "cluster_label" in gaps_out.columns:
        cluster_label_map = (
            gaps_out[[cluster_col, "cluster_label"]]
            .dropna()
            .drop_duplicates(subset=[cluster_col])
            .set_index(cluster_col)["cluster_label"]
            .to_dict()
        )
        cluster_overall_summary["cluster_label"] = cluster_overall_summary[cluster_col].map(cluster_label_map)
    else:
        cluster_overall_summary["cluster_label"] = cluster_overall_summary[cluster_col].map(
            approved.groupby(cluster_col)["hotspot_unit"].agg(lambda s: s.mode().iloc[0] if not s.mode().empty else str(s.iloc[0])).to_dict()
        )

    cluster_overall_summary["mu_departures_per_hour"] = 60.0 / (cluster_overall_summary["mean_dwell_minutes"] + EPS)

    # -------------------------
    # Cluster + vehicle-type summary
    # -------------------------
    cluster_type_summary = (
        gaps_out.groupby([cluster_col, vehicle_type_col], dropna=False)
        .agg(
            gap_count=("gap_minutes", "size"),
            unique_vehicles=(vehicle_col, "nunique"),
            mean_dwell_minutes=("gap_minutes", "mean"),
            median_dwell_minutes=("gap_minutes", "median"),
            std_dwell_minutes=("gap_minutes", lambda s: float(s.std(ddof=0)) if len(s) else 0.0),
        )
        .reset_index()
        .rename(columns={vehicle_type_col: "vehicle_type"})
        .sort_values([cluster_col, "gap_count", "mean_dwell_minutes"], ascending=[True, False, False])
    )
    cluster_type_summary["mu_departures_per_hour"] = 60.0 / (cluster_type_summary["mean_dwell_minutes"] + EPS)

    # -------------------------
    # Wide per-cluster summary for Stage 5 convenience
    # -------------------------
    type_wide = cluster_type_summary.pivot_table(
        index=cluster_col,
        columns="vehicle_type",
        values="mean_dwell_minutes",
        aggfunc="mean"
    )
    type_wide.columns = [f"mean_dwell_minutes__{str(c).replace(' ', '_').upper()}" for c in type_wide.columns]
    type_wide = type_wide.reset_index()

    type_wide_mu = cluster_type_summary.pivot_table(
        index=cluster_col,
        columns="vehicle_type",
        values="mu_departures_per_hour",
        aggfunc="mean"
    )
    type_wide_mu.columns = [f"mu_departures_per_hour__{str(c).replace(' ', '_').upper()}" for c in type_wide_mu.columns]
    type_wide_mu = type_wide_mu.reset_index()

    cluster_wide_summary = cluster_overall_summary.merge(type_wide, on=cluster_col, how="left").merge(
        type_wide_mu, on=cluster_col, how="left"
    )

    # -------------------------
    # Merge back onto records for later stages
    # -------------------------
    merged = approved.merge(
        cluster_overall_summary[[cluster_col, "gap_count", "mean_dwell_minutes", "median_dwell_minutes", "mu_departures_per_hour"]],
        on=cluster_col,
        how="left"
    )

    # -------------------------
    # Save outputs
    # -------------------------
    gaps_out.to_csv(OUT_DIR / "phase4_gap_event_log.csv", index=False)
    overall_summary.to_csv(OUT_DIR / "phase4_overall_mu_summary.csv", index=False)
    vehicle_type_summary.to_csv(OUT_DIR / "phase4_vehicle_type_mu_summary.csv", index=False)
    cluster_overall_summary.to_csv(OUT_DIR / "phase4_cluster_mu_summary.csv", index=False)
    cluster_type_summary.to_csv(OUT_DIR / "phase4_cluster_vehicle_type_mu_summary.csv", index=False)
    cluster_wide_summary.to_csv(OUT_DIR / "phase4_cluster_mu_summary_wide.csv", index=False)
    merged.to_csv(OUT_DIR / "phase4_merged_with_prior_scores.csv", index=False)

    # For convenience in Stage 5
    cluster_overall_summary[[cluster_col, "cluster_label", "mean_dwell_minutes", "mu_departures_per_hour", "gap_count"]].to_csv(
        OUT_DIR / "phase4_stage5_mu_lookup.csv", index=False
    )

    # -------------------------
    # Console summary
    # -------------------------
    print("Phase 4 complete")
    print("Input source:", src)
    print("Approved rows used:", len(approved))
    print("Valid inter-record gaps:", len(gaps_out))
    print("Overall mean dwell (min):", round(overall_mean_dwell, 3))
    print("Overall μ (departures/hour):", round(overall_mu, 6))
    print("Clusters with dwell estimates:", len(cluster_overall_summary))
    print("Vehicle types with dwell estimates:", len(vehicle_type_summary))
    print("Outputs saved to:", OUT_DIR.resolve())

    print("\nVehicle-type summary:")
    print(
        vehicle_type_summary[[
            "vehicle_type", "gap_count", "mean_dwell_minutes", "mu_departures_per_hour"
        ]].to_string(index=False)
    )

    print("\nTop clusters by gap count:")
    print(
        cluster_overall_summary[[
            cluster_col, "cluster_label", "gap_count", "mean_dwell_minutes", "mu_departures_per_hour"
        ]].sort_values("gap_count", ascending=False).head(10).to_string(index=False)
    )

    # -------------------------
    # Plots
    # -------------------------
    plt.figure(figsize=(12, 6))
    top_clusters = cluster_overall_summary.sort_values("gap_count", ascending=True).tail(TOP_N)
    plt.barh(top_clusters["cluster_label"], top_clusters["mean_dwell_minutes"])
    plt.title("Top Clusters by Mean Dwell Time")
    plt.xlabel("Mean dwell time (minutes)")
    plt.ylabel("Cluster")
    save_plot(OUT_DIR / "top_clusters_mean_dwell.png")

    plt.figure(figsize=(12, 6))
    top_types = vehicle_type_summary.sort_values("gap_count", ascending=True).tail(TOP_N)
    plt.barh(top_types["vehicle_type"], top_types["mean_dwell_minutes"])
    plt.title("Vehicle-Type Mean Dwell Time")
    plt.xlabel("Mean dwell time (minutes)")
    plt.ylabel("Vehicle type")
    save_plot(OUT_DIR / "vehicle_type_mean_dwell.png")

    plt.figure(figsize=(10, 5))
    plt.hist(gaps_out["gap_minutes"], bins=40)
    plt.title("Distribution of Inter-Record Dwell Gaps")
    plt.xlabel("Gap minutes")
    plt.ylabel("Frequency")
    save_plot(OUT_DIR / "gap_distribution.png")

if __name__ == "__main__":
    main()

Loaded: phase3_outputs_1/phase3_clustered_dataset.csv
Phase 4 complete
Input source: phase3_outputs_1/phase3_clustered_dataset.csv
Approved rows used: 92332
Valid inter-record gaps: 2122
Overall mean dwell (min): 98.041
Overall μ (departures/hour): 0.611989
Clusters with dwell estimates: 239
Vehicle types with dwell estimates: 19
Outputs saved to: /content/phase4_outputs_1

Vehicle-type summary:
       vehicle_type  gap_count  mean_dwell_minutes  mu_departures_per_hour
            SCOOTER        799          110.365457                0.543648
                CAR        711           95.087201                0.631000
        MOTOR CYCLE        296           98.131757                0.611423
     PASSENGER AUTO        181           53.933702                1.112477
           MAXI-CAB         52           93.576923                0.641184
                LGV         25           83.160000                0.721501
              MOPED         13           46.230769                1.297837
 

In [ ]:
!pip install osmnx networkx

In [ ]:
# =========================
# Phase 5 — Queueing, Capacity Loss, and Delay Estimation
# Stage 5a: M/M/∞ Queue Model
# Stage 5b: Zhao Capacity Loss Model
# Stage 5c: Modified BPR Delay Model
# Also prepares growth and criticality inputs for Stage 6
# =========================

import ast
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# -------------------------
# Config
# -------------------------
PHASE4_MERGED_PATH = Path("phase4_outputs_1/phase4_merged_with_prior_scores.csv")
PHASE4_CLUSTER_PATH = Path("phase4_outputs_1/phase4_cluster_mu_summary.csv")
PHASE4_GAP_PATH = Path("phase4_outputs_1/phase4_gap_event_log.csv")

OUT_DIR = Path("phase5_outputs_1")
OUT_DIR.mkdir(parents=True, exist_ok=True)

EPS = 1e-9
TOP_N = 25

# Doc / paper constants
ALPHA_BPR = 0.56
BETA_BPR = 2.12

# Default geometric assumptions when OSMnx data is unavailable
DEFAULT_LANES = 2
DEFAULT_WIDTH_PER_LANE_M = 3.5
DEFAULT_CARRIAGEWAY_WIDTH_M = DEFAULT_LANES * DEFAULT_WIDTH_PER_LANE_M
DEFAULT_LINK_LENGTH_M = 250.0

ROAD_CLASS_PRIORITY = [
    "motorway", "trunk", "primary", "secondary", "tertiary",
    "unclassified", "residential", "living_street", "service", "road"
]

ROAD_CLASS_SPEED_KMH = {
    "motorway": 60.0,
    "trunk": 55.0,
    "primary": 45.0,
    "secondary": 40.0,
    "tertiary": 35.0,
    "unclassified": 30.0,
    "residential": 30.0,
    "living_street": 20.0,
    "service": 25.0,
    "road": 30.0,
}

ROAD_CLASS_BASE_CAPACITY_PER_LANE = {
    "motorway": 2200.0,
    "trunk": 2100.0,
    "primary": 1900.0,
    "secondary": 1800.0,
    "tertiary": 1700.0,
    "unclassified": 1650.0,
    "residential": 1500.0,
    "living_street": 1200.0,
    "service": 1400.0,
    "road": 1600.0,
}

# Typical blocking widths by vehicle class
VEHICLE_WIDTH_M = {
    "SCOOTER": 0.80,
    "MOTOR CYCLE": 0.90,
    "MOTORCYCLE": 0.90,
    "BICYCLE": 0.60,
    "CYCLE": 0.60,
    "PASSENGER AUTO": 1.60,
    "AUTO": 1.60,
    "CAR": 1.90,
    "SUV": 2.00,
    "JEEP": 2.00,
    "VAN": 2.20,
    "TEMPO": 2.20,
    "BUS": 2.60,
    "TRUCK": 2.60,
    "LORRY": 2.60,
    "TANKER": 2.80,
    "TRACTOR": 2.20,
    "MINI TRUCK": 2.20,
    "AMBULANCE": 2.00,
    "UNKNOWN": 1.90,
}

# -------------------------
# Helpers
# -------------------------
def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def safe_ratio(a, b):
    return a / (b + EPS)

def minmax(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce").fillna(0.0).astype(float)
    if s.nunique(dropna=True) <= 1:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.min()) / (s.max() - s.min() + EPS)

def parse_listlike(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    s = str(value).strip()
    if not s:
        return []
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, (list, tuple)):
            return list(parsed)
        return [parsed]
    except Exception:
        if s.startswith("[") and s.endswith("]"):
            s = s[1:-1]
        parts = [p.strip().strip("'").strip('"') for p in s.split(",")]
        return [p for p in parts if p]

def parse_datetime_ist(df: pd.DataFrame) -> pd.Series:
    if "created_datetime_ist" in df.columns:
        ts = pd.to_datetime(df["created_datetime_ist"], errors="coerce", utc=True)
        if ts.notna().any():
            return ts.dt.tz_convert("Asia/Kolkata")
        ts = pd.to_datetime(df["created_datetime_ist"], errors="coerce")
        return ts.dt.tz_localize("Asia/Kolkata", nonexistent="NaT", ambiguous="NaT")

    if "created_datetime_parsed" in df.columns:
        ts = pd.to_datetime(df["created_datetime_parsed"], errors="coerce", utc=True)
        return ts.dt.tz_convert("Asia/Kolkata")

    if "created_datetime" not in df.columns:
        raise ValueError("Missing created_datetime / created_datetime_ist column.")

    ts = pd.to_datetime(df["created_datetime"], errors="coerce", utc=True)
    return ts.dt.tz_convert("Asia/Kolkata")

def week_start_monday(series):
    dt = pd.to_datetime(series, errors="coerce", utc=True).dt.tz_convert("Asia/Kolkata")
    week_start = dt.dt.normalize() - pd.to_timedelta(dt.dt.weekday, unit="D")
    return week_start.dt.tz_localize(None)

def make_hotspot_unit(row):
    junction = clean_text(row.get("junction_name", ""))
    if junction and junction.upper() != "NO JUNCTION":
        return f"JUNCTION::{junction}"
    station = clean_text(row.get("police_station", "UNKNOWN"))
    if not station:
        station = "UNKNOWN"
    return f"POLICE_STATION::{station}"

def get_cluster_col(df: pd.DataFrame) -> str:
    for c in ["st_dbscan_cluster_id", "cluster_id", "dbscan_cluster_id"]:
        if c in df.columns:
            return c
    raise ValueError("No cluster id column found.")

def get_vehicle_col(df: pd.DataFrame) -> str:
    if "updated_vehicle_number" in df.columns and "vehicle_number" in df.columns:
        upd = df["updated_vehicle_number"].fillna("").astype(str).str.strip()
        orig = df["vehicle_number"].fillna("").astype(str).str.strip()
        df["canonical_vehicle_number"] = np.where(upd.ne(""), upd, orig)
        return "canonical_vehicle_number"
    if "vehicle_number" not in df.columns:
        raise ValueError("Missing vehicle_number / updated_vehicle_number.")
    df["canonical_vehicle_number"] = df["vehicle_number"].fillna("").astype(str).str.strip()
    return "canonical_vehicle_number"

def get_vehicle_type_col(df: pd.DataFrame) -> str:
    if "updated_vehicle_type" in df.columns and "vehicle_type" in df.columns:
        upd = df["updated_vehicle_type"].fillna("").astype(str).str.strip()
        orig = df["vehicle_type"].fillna("").astype(str).str.strip()
        df["canonical_vehicle_type"] = np.where(upd.ne(""), upd, orig)
        return "canonical_vehicle_type"
    if "vehicle_type" in df.columns:
        df["canonical_vehicle_type"] = df["vehicle_type"].fillna("UNKNOWN").astype(str).str.strip()
        df.loc[df["canonical_vehicle_type"].eq(""), "canonical_vehicle_type"] = "UNKNOWN"
        return "canonical_vehicle_type"
    df["canonical_vehicle_type"] = "UNKNOWN"
    return "canonical_vehicle_type"

def normalize_road_class(highway_value):
    vals = parse_listlike(highway_value)
    if not vals:
        raw = clean_text(highway_value).lower()
        if not raw:
            return "road"
        vals = [raw]

    normalized = []
    for v in vals:
        s = clean_text(v).lower()
        s = s.replace(" ", "_")
        if s.endswith("_link"):
            s = s.replace("_link", "")
        if s in {"motorway", "trunk", "primary", "secondary", "tertiary", "unclassified", "residential", "living_street", "service", "road"}:
            normalized.append(s)
        elif s in {"motorway_link"}:
            normalized.append("motorway")
        elif s in {"trunk_link"}:
            normalized.append("trunk")
        elif s in {"primary_link"}:
            normalized.append("primary")
        elif s in {"secondary_link"}:
            normalized.append("secondary")
        elif s in {"tertiary_link"}:
            normalized.append("tertiary")
        else:
            normalized.append("road")

    for cls in ROAD_CLASS_PRIORITY:
        if cls in normalized:
            return cls
    return normalized[0] if normalized else "road"

def parse_lanes(value):
    s = clean_text(value)
    if not s:
        return None
    s = s.replace(";", ",").replace("|", ",")
    parts = [p.strip() for p in s.split(",") if p.strip()]
    nums = []
    for p in parts:
        try:
            nums.append(float(p))
        except Exception:
            pass
    if nums:
        return max(1, int(round(nums[0])))
    try:
        return max(1, int(round(float(s))))
    except Exception:
        return None

def parse_width(value):
    s = clean_text(value)
    if not s:
        return None
    s = s.lower().replace("m", "").strip()
    s = s.replace(",", ".")
    try:
        return float(s)
    except Exception:
        return None

def road_speed_kmh(road_class):
    return ROAD_CLASS_SPEED_KMH.get(road_class, 30.0)

def base_capacity_per_lane(road_class):
    return ROAD_CLASS_BASE_CAPACITY_PER_LANE.get(road_class, 1600.0)

def vehicle_width_m(vehicle_type):
    vt = clean_text(vehicle_type).upper()
    return VEHICLE_WIDTH_M.get(vt, 1.90)

def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000.0
    p1 = np.radians(lat1)
    p2 = np.radians(lat2)
    dp = np.radians(lat2 - lat1)
    dl = np.radians(lon2 - lon1)
    a = np.sin(dp / 2.0) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2.0) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

def clamp(x, lo, hi):
    return max(lo, min(hi, x))

def save_plot(path):
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()

# -------------------------
# Load data
# -------------------------
if PHASE4_MERGED_PATH.exists():
    df = pd.read_csv(PHASE4_MERGED_PATH, low_memory=False)
else:
    raise FileNotFoundError(
        "Phase 4 merged output not found. Expected phase4_outputs/phase4_merged_with_prior_scores.csv"
    )

cluster_col = get_cluster_col(df)
vehicle_col = get_vehicle_col(df)
vehicle_type_col = get_vehicle_type_col(df)

# Timestamp and baseline columns
df["created_datetime_ist"] = parse_datetime_ist(df)
df["created_datetime_ist_naive"] = df["created_datetime_ist"].dt.tz_localize(None)
df["week_start"] = week_start_monday(df["created_datetime"])
df["cluster_week_start"] = df["created_datetime_ist"].dt.to_period("W-MON").dt.start_time

if "validation_status_clean" in df.columns:
    df["validation_status_clean"] = df["validation_status_clean"].fillna("").astype(str).str.lower()
elif "validation_status" in df.columns:
    df["validation_status_clean"] = df["validation_status"].fillna("").astype(str).str.lower()
else:
    df["validation_status_clean"] = "approved"

if "severity_score" not in df.columns:
    df["severity_score"] = 1.0

if "hotspot_unit" not in df.columns:
    if {"junction_name", "police_station"}.issubset(df.columns):
        df["hotspot_unit"] = df.apply(make_hotspot_unit, axis=1)
    else:
        df["hotspot_unit"] = "UNKNOWN"

approved = df.loc[df["validation_status_clean"].eq("approved")].copy()
approved = approved.dropna(subset=[cluster_col, "created_datetime_ist_naive", "latitude", "longitude"]).copy()
approved = approved.loc[approved[cluster_col].ne(-1)].copy()
approved = approved.loc[approved[vehicle_col].fillna("").astype(str).str.strip().ne("")].copy()
approved = approved.reset_index(drop=True)

if approved.empty:
    raise ValueError("No approved clustered rows available for Stage 5.")

print("Approved clustered rows:", len(approved))
print("Clusters:", approved[cluster_col].nunique())

# -------------------------
# Stage 5a — Arrival rate λ
# -------------------------
peak_window_mask = approved["created_datetime_ist"].dt.hour.between(8, 12, inclusive="both")
peak_window = approved.loc[peak_window_mask].copy()

cluster_lambda = (
    approved.groupby(cluster_col)
    .agg(
        records_total=("id", "size") if "id" in approved.columns else (vehicle_col, "size"),
        records_peak_window=(vehicle_col, lambda s: int(peak_window_mask.loc[s.index].sum())),
        records_peak_hour=(vehicle_col, lambda s: int(
            approved.loc[s.index]
            .groupby(approved.loc[s.index, "created_datetime_ist"].dt.hour)
            .size()
            .max()
            if len(s.index) > 0 else 0
        )),
        first_seen=("created_datetime_ist_naive", "min"),
        last_seen=("created_datetime_ist_naive", "max"),
        centroid_lat=("latitude", "mean"),
        centroid_lon=("longitude", "mean"),
        severity_sum=("severity_score", "sum"),
        severity_mean=("severity_score", "mean"),
        unique_vehicles=(vehicle_col, "nunique"),
        unique_vehicle_types=(vehicle_type_col, "nunique"),
    )
    .reset_index()
)

cluster_lambda["lambda_hr_peak_window"] = cluster_lambda["records_peak_window"] / 5.0
cluster_lambda["lambda_hr_peak_hour"] = cluster_lambda["records_peak_hour"].astype(float)

# -------------------------
# Stage 4 carry-over — μ / dwell time
# -------------------------
mu_source = None
if PHASE4_CLUSTER_PATH.exists():
    mu_source = pd.read_csv(PHASE4_CLUSTER_PATH, low_memory=False)
    if cluster_col not in mu_source.columns and "st_dbscan_cluster_id" in mu_source.columns:
        mu_source = mu_source.rename(columns={"st_dbscan_cluster_id": cluster_col})

    keep_cols = [cluster_col]
    for c in ["cluster_label", "gap_count", "mean_dwell_minutes", "median_dwell_minutes", "std_dwell_minutes", "mu_departures_per_hour"]:
        if c in mu_source.columns:
            keep_cols.append(c)
    mu_source = mu_source[keep_cols].drop_duplicates(cluster_col)
else:
    # Fallback: derive from the merged data if the dedicated Stage 4 file is missing
    tmp = approved.sort_values([cluster_col, vehicle_col, "cluster_week_start", "created_datetime_ist_naive"]).copy()
    tmp["prev_ts"] = tmp.groupby([cluster_col, vehicle_col, tmp["created_datetime_ist"].dt.date])["created_datetime_ist_naive"].shift(1)
    tmp["gap_minutes"] = (tmp["created_datetime_ist_naive"] - tmp["prev_ts"]).dt.total_seconds() / 60.0
    tmp = tmp.loc[tmp["gap_minutes"].notna() & (tmp["gap_minutes"] > 0)].copy()
    mu_source = (
        tmp.groupby(cluster_col)
        .agg(
            gap_count=("gap_minutes", "size"),
            mean_dwell_minutes=("gap_minutes", "mean"),
            median_dwell_minutes=("gap_minutes", "median"),
            std_dwell_minutes=("gap_minutes", lambda s: float(s.std(ddof=0)) if len(s) else 0.0),
        )
        .reset_index()
    )
    mu_source["mu_departures_per_hour"] = 60.0 / (mu_source["mean_dwell_minutes"] + EPS)

cluster_table = cluster_lambda.merge(mu_source, on=cluster_col, how="left")

if "mu_departures_per_hour" not in cluster_table.columns:
    cluster_table["mu_departures_per_hour"] = 60.0 / (cluster_table["mean_dwell_minutes"] + EPS)

cluster_table["mean_dwell_minutes"] = pd.to_numeric(cluster_table["mean_dwell_minutes"], errors="coerce")
cluster_table["mu_departures_per_hour"] = pd.to_numeric(cluster_table["mu_departures_per_hour"], errors="coerce")
cluster_table["mu_departures_per_hour"] = cluster_table["mu_departures_per_hour"].fillna(
    60.0 / (cluster_table["mean_dwell_minutes"] + EPS)
)

cluster_table["blocking_vehicles_L"] = cluster_table["lambda_hr_peak_window"] / (cluster_table["mu_departures_per_hour"] + EPS)

# -------------------------
# Vehicle mix and blocking width
# -------------------------
vehicle_mix = (
    approved.groupby([cluster_col, vehicle_type_col])
    .size()
    .reset_index(name="count")
    .rename(columns={vehicle_type_col: "vehicle_type"})
)

vehicle_mix["vehicle_type"] = vehicle_mix["vehicle_type"].fillna("UNKNOWN").astype(str).str.strip()
vehicle_mix["vehicle_width_m"] = vehicle_mix["vehicle_type"].apply(vehicle_width_m)

cluster_width = (
    vehicle_mix.groupby(cluster_col)
    .apply(lambda g: safe_ratio((g["count"] * g["vehicle_width_m"]).sum(), g["count"].sum()))
    .reset_index(name="vehicle_width_avg_m")
)

cluster_table = cluster_table.merge(cluster_width, on=cluster_col, how="left")
cluster_table["vehicle_width_avg_m"] = cluster_table["vehicle_width_avg_m"].fillna(1.90)

# -------------------------
# Trend acceleration / growth multiplier
# -------------------------
weekly_counts = (
    approved.groupby([cluster_col, "cluster_week_start"])
    .size()
    .reset_index(name="weekly_count")
    .sort_values([cluster_col, "cluster_week_start"])
)

growth_rows = []
for cid, g in weekly_counts.groupby(cluster_col):
    counts = g.sort_values("cluster_week_start")["weekly_count"].to_numpy(dtype=float)
    if len(counts) < 2:
        first_mean = float(counts.mean()) if len(counts) else 0.0
        second_mean = first_mean
        growth_pct = 0.0
    else:
        mid = max(1, len(counts) // 2)
        first_half = counts[:mid]
        second_half = counts[mid:]
        first_mean = float(first_half.mean()) if len(first_half) else 0.0
        second_mean = float(second_half.mean()) if len(second_half) else 0.0
        if first_mean <= 0:
            growth_pct = second_mean
        else:
            growth_pct = (second_mean - first_mean) / (first_mean + EPS)

    growth_multiplier = max(0.5, 1.0 + max(0.0, growth_pct))
    growth_rows.append({
        cluster_col: cid,
        "growth_first_half_mean": first_mean,
        "growth_second_half_mean": second_mean,
        "growth_pct_change": growth_pct,
        "growth_multiplier": growth_multiplier,
    })

growth_df = pd.DataFrame(growth_rows)

# -------------------------
# Road-network context via OSMnx (optional, graceful fallback)
# -------------------------
road_context_rows = []

try:
    import osmnx as ox
    import networkx as nx

    HAS_OSMNX = True
except Exception:
    HAS_OSMNX = False

def pick_edge_data(edge_data):
    highway = edge_data.get("highway", None)
    road_class = normalize_road_class(highway)

    lanes = parse_lanes(edge_data.get("lanes", None))
    width = parse_width(edge_data.get("width", None))
    length_m = edge_data.get("length", None)

    if lanes is None:
        lanes = DEFAULT_LANES
    if width is None:
        width = lanes * DEFAULT_WIDTH_PER_LANE_M
    if length_m is None:
        length_m = DEFAULT_LINK_LENGTH_M

    return road_class, int(lanes), float(width), float(length_m)

def enrich_cluster_geometry(lat, lon):
    """
    Returns road/network context for a cluster centroid.
    If OSMnx is unavailable or fails, returns sensible fallbacks.
    """
    result = {
        "geometry_source": "fallback",
        "osmnx_node_id": np.nan,
        "osmnx_edge_u": np.nan,
        "osmnx_edge_v": np.nan,
        "osmnx_edge_key": np.nan,
        "road_class": "road",
        "lane_count": DEFAULT_LANES,
        "carriageway_width_m": DEFAULT_CARRIAGEWAY_WIDTH_M,
        "link_length_m": DEFAULT_LINK_LENGTH_M,
        "junction_degree": 4,
        "betweenness_centrality": 0.0,
    }

    if not HAS_OSMNX:
        return result

    try:
        # Small local drive graph around the hotspot centroid
        G = ox.graph_from_point((lat, lon), dist=750, network_type="drive", simplify=True, retain_all=False)

        # nearest node / edge
        nearest_node = ox.distance.nearest_nodes(G, X=lon, Y=lat)
        nearest_edge = ox.distance.nearest_edges(G, X=lon, Y=lat)

        u, v, k = nearest_edge
        edge_data = G.edges[u, v, k]

        road_class, lanes, width, length_m = pick_edge_data(edge_data)

        UG = nx.Graph(G)
        # Centrality in the local graph
        bc = nx.betweenness_centrality(UG, normalized=True, weight="length")
        centrality = float(bc.get(nearest_node, 0.0))
        degree = int(UG.degree[nearest_node])

        result.update({
            "geometry_source": "osmnx",
            "osmnx_node_id": nearest_node,
            "osmnx_edge_u": u,
            "osmnx_edge_v": v,
            "osmnx_edge_key": k,
            "road_class": road_class,
            "lane_count": lanes,
            "carriageway_width_m": width,
            "link_length_m": length_m,
            "junction_degree": degree,
            "betweenness_centrality": centrality,
        })
        return result

    except Exception:
        return result

for _, row in cluster_table.iterrows():
    road_context_rows.append({
        cluster_col: row[cluster_col],
        **enrich_cluster_geometry(float(row["centroid_lat"]), float(row["centroid_lon"]))
    })

road_context_df = pd.DataFrame(road_context_rows)

cluster_table = cluster_table.merge(road_context_df, on=cluster_col, how="left")
cluster_table["lane_count"] = cluster_table["lane_count"].fillna(DEFAULT_LANES).astype(int)
cluster_table["carriageway_width_m"] = cluster_table["carriageway_width_m"].fillna(DEFAULT_CARRIAGEWAY_WIDTH_M).astype(float)
cluster_table["link_length_m"] = cluster_table["link_length_m"].fillna(DEFAULT_LINK_LENGTH_M).astype(float)
cluster_table["junction_degree"] = cluster_table["junction_degree"].fillna(4).astype(float)
cluster_table["betweenness_centrality"] = cluster_table["betweenness_centrality"].fillna(0.0).astype(float)

# Normalize criticality across all clusters
cluster_table["junction_degree_norm"] = minmax(cluster_table["junction_degree"])
cluster_table["betweenness_centrality_norm"] = minmax(cluster_table["betweenness_centrality"])
cluster_table["criticality_factor"] = 1.0 + 0.5 * cluster_table["junction_degree_norm"] + 0.5 * cluster_table["betweenness_centrality_norm"]

# -------------------------
# Stage 5b — Zhao capacity reduction
# -------------------------
cluster_table["base_saturation_per_lane_pcu_hr"] = cluster_table["road_class"].map(base_capacity_per_lane).fillna(1600.0)
cluster_table["free_flow_speed_kmh"] = cluster_table["road_class"].apply(road_speed_kmh)
cluster_table["free_flow_time_min"] = (
    cluster_table["link_length_m"] * 60.0 / (cluster_table["free_flow_speed_kmh"] * 1000.0 + EPS)
)

# Base capacity
cluster_table["width_factor"] = cluster_table["carriageway_width_m"] / (cluster_table["lane_count"] * DEFAULT_WIDTH_PER_LANE_M + EPS)
cluster_table["width_factor"] = cluster_table["width_factor"].clip(0.70, 1.20)

cluster_table["base_capacity_pcu_hr"] = (
    cluster_table["base_saturation_per_lane_pcu_hr"] *
    cluster_table["lane_count"] *
    cluster_table["width_factor"]
)

# Effective blocked-width fraction
cluster_table["blocked_width_fraction_raw"] = (
    cluster_table["blocking_vehicles_L"] * cluster_table["vehicle_width_avg_m"]
) / (cluster_table["carriageway_width_m"] + EPS)

cluster_table["blocked_width_fraction"] = cluster_table["blocked_width_fraction_raw"].clip(0.0, 0.95)

cluster_table["reduced_capacity_pcu_hr"] = (
    cluster_table["base_capacity_pcu_hr"] * (1.0 - cluster_table["blocked_width_fraction"])
).clip(lower=cluster_table["base_capacity_pcu_hr"] * 0.10)

cluster_table["capacity_loss_pct"] = 1.0 - safe_ratio(cluster_table["reduced_capacity_pcu_hr"], cluster_table["base_capacity_pcu_hr"])

# -------------------------
# Stage 5c — Modified BPR delay
# -------------------------
cluster_table["V_over_C0"] = cluster_table["lambda_hr_peak_window"] / (cluster_table["base_capacity_pcu_hr"] + EPS)
cluster_table["V_over_Cp"] = cluster_table["lambda_hr_peak_window"] / (cluster_table["reduced_capacity_pcu_hr"] + EPS)

cluster_table["delay_minutes_per_vehicle"] = (
    cluster_table["free_flow_time_min"] * ALPHA_BPR *
    (np.power(cluster_table["V_over_Cp"], BETA_BPR) - np.power(cluster_table["V_over_C0"], BETA_BPR))
)

cluster_table["delay_minutes_per_vehicle"] = cluster_table["delay_minutes_per_vehicle"].clip(lower=0.0)

# Optional composite inputs for Stage 6
cluster_table = cluster_table.merge(growth_df, on=cluster_col, how="left")
cluster_table["growth_multiplier"] = cluster_table["growth_multiplier"].fillna(1.0)
cluster_table["growth_pct_change"] = cluster_table["growth_pct_change"].fillna(0.0)

# -------------------------
# Chronic offender list (for the architecture branch)
# -------------------------
vehicle_offenders = (
    approved.groupby(vehicle_col)
    .agg(
        total_violations=(vehicle_col, "size"),
        unique_clusters=(cluster_col, "nunique"),
        unique_hotspots=("hotspot_unit", "nunique"),
        first_seen=("created_datetime_ist_naive", "min"),
        last_seen=("created_datetime_ist_naive", "max"),
        dominant_vehicle_type=(vehicle_type_col, lambda s: s.mode().iloc[0] if not s.mode().empty else "UNKNOWN"),
    )
    .reset_index()
    .rename(columns={vehicle_col: "vehicle_number"})
    .sort_values(["total_violations", "unique_clusters"], ascending=[False, False])
)
vehicle_offenders["chronic_offender_flag"] = (vehicle_offenders["total_violations"] >= 5).astype(int)
chronic_offenders = vehicle_offenders.loc[vehicle_offenders["chronic_offender_flag"].eq(1)].copy()

# Cluster-level repeat offender counts
repeat_vehicle_counts = (
    approved.groupby([cluster_col, vehicle_col])
    .size()
    .reset_index(name="vehicle_cluster_count")
)
repeat_vehicle_counts["repeat_flag_2plus"] = (repeat_vehicle_counts["vehicle_cluster_count"] >= 2).astype(int)
repeat_vehicle_counts["chronic_flag_5plus"] = (repeat_vehicle_counts["vehicle_cluster_count"] >= 5).astype(int)

cluster_repeat_summary = (
    repeat_vehicle_counts.groupby(cluster_col)
    .agg(
        repeat_vehicle_count_2plus=("repeat_flag_2plus", "sum"),
        chronic_vehicle_count_5plus=("chronic_flag_5plus", "sum"),
    )
    .reset_index()
)

cluster_table = cluster_table.merge(cluster_repeat_summary, on=cluster_col, how="left")
cluster_table["repeat_vehicle_count_2plus"] = cluster_table["repeat_vehicle_count_2plus"].fillna(0).astype(int)
cluster_table["chronic_vehicle_count_5plus"] = cluster_table["chronic_vehicle_count_5plus"].fillna(0).astype(int)

# -------------------------
# Final Stage 5 priority view
# -------------------------
priority_cols = [
    cluster_col, "cluster_label", "records_total", "records_peak_window", "lambda_hr_peak_window",
    "records_peak_hour", "lambda_hr_peak_hour", "mean_dwell_minutes", "mu_departures_per_hour",
    "blocking_vehicles_L", "road_class", "lane_count", "carriageway_width_m", "link_length_m",
    "base_capacity_pcu_hr", "reduced_capacity_pcu_hr", "capacity_loss_pct", "free_flow_speed_kmh",
    "free_flow_time_min", "delay_minutes_per_vehicle", "junction_degree", "betweenness_centrality",
    "criticality_factor", "growth_pct_change", "growth_multiplier", "vehicle_width_avg_m",
    "dominant_vehicle_type", "unique_vehicles", "unique_vehicle_types",
    "repeat_vehicle_count_2plus", "chronic_vehicle_count_5plus", "geometry_source"
]
for c in priority_cols:
    if c not in cluster_table.columns:
        cluster_table[c] = np.nan

priority_table = cluster_table[priority_cols].copy()
priority_table = priority_table.sort_values(
    ["delay_minutes_per_vehicle", "lambda_hr_peak_window", "blocking_vehicles_L"],
    ascending=[False, False, False]
).reset_index(drop=True)
priority_table["stage5_rank"] = np.arange(1, len(priority_table) + 1)

# Helpful normalized versions for later stages
priority_table["delay_minutes_norm"] = minmax(priority_table["delay_minutes_per_vehicle"])
priority_table["lambda_norm"] = minmax(priority_table["lambda_hr_peak_window"])
priority_table["severity_sum_norm"] = minmax(priority_table["records_total"])

# -------------------------
# Merge back onto records for Stage 6
# -------------------------
stage5_record_view = approved.merge(
    priority_table[[
        cluster_col, "cluster_label", "records_total", "lambda_hr_peak_window", "mean_dwell_minutes",
        "mu_departures_per_hour", "blocking_vehicles_L", "road_class", "lane_count",
        "carriageway_width_m", "link_length_m", "base_capacity_pcu_hr", "reduced_capacity_pcu_hr",
        "capacity_loss_pct", "free_flow_time_min", "delay_minutes_per_vehicle",
        "junction_degree", "betweenness_centrality", "criticality_factor",
        "growth_pct_change", "growth_multiplier", "stage5_rank"
    ]],
    on=cluster_col,
    how="left",
    suffixes=("", "_stage5")
)

# -------------------------
# Save outputs
# -------------------------
cluster_table.to_csv(OUT_DIR / "phase5_cluster_capacity_loss.csv", index=False)
priority_table.to_csv(OUT_DIR / "phase5_priority_table.csv", index=False)
stage5_record_view.to_csv(OUT_DIR / "phase5_enriched_records.csv", index=False)
vehicle_mix.to_csv(OUT_DIR / "phase5_vehicle_mix.csv", index=False)
weekly_counts.to_csv(OUT_DIR / "phase5_weekly_cluster_counts.csv", index=False)
growth_df.to_csv(OUT_DIR / "phase5_growth_summary.csv", index=False)
chronic_offenders.to_csv(OUT_DIR / "phase5_chronic_offenders.csv", index=False)
road_context_df.to_csv(OUT_DIR / "phase5_road_context.csv", index=False)

# Compact Stage 5 handoff tables
priority_table[[
    "stage5_rank", cluster_col, "cluster_label", "delay_minutes_per_vehicle",
    "lambda_hr_peak_window", "mu_departures_per_hour", "blocking_vehicles_L",
    "capacity_loss_pct", "criticality_factor", "growth_multiplier", "road_class",
    "lane_count", "carriageway_width_m", "free_flow_time_min"
]].to_csv(OUT_DIR / "phase5_stage5_handoff.csv", index=False)

# -------------------------
# Console summary
# -------------------------
print("Phase 5 complete")
print("Clusters scored:", len(priority_table))
print("Chronic offenders:", len(chronic_offenders))
print("OSMnx available:", HAS_OSMNX)
print("Outputs saved to:", OUT_DIR.resolve())

print("\nTop 10 clusters by delay:")
print(
    priority_table[[
        "stage5_rank", cluster_col, "cluster_label", "delay_minutes_per_vehicle",
        "lambda_hr_peak_window", "mu_departures_per_hour", "blocking_vehicles_L",
        "capacity_loss_pct", "criticality_factor", "growth_multiplier"
    ]].head(10).to_string(index=False)
)

# -------------------------
# Plots
# -------------------------
plt.figure(figsize=(12, 7))
top_delay = priority_table.head(TOP_N).sort_values("delay_minutes_per_vehicle", ascending=True)
plt.barh(top_delay["cluster_label"], top_delay["delay_minutes_per_vehicle"])
plt.title("Top Clusters by Delay Minutes per Vehicle")
plt.xlabel("Delay minutes per vehicle")
plt.ylabel("Cluster")
save_plot(OUT_DIR / "top_clusters_delay.png")

plt.figure(figsize=(12, 7))
top_capacity = priority_table.head(TOP_N).sort_values("capacity_loss_pct", ascending=True)
plt.barh(top_capacity["cluster_label"], top_capacity["capacity_loss_pct"])
plt.title("Top Clusters by Capacity Loss")
plt.xlabel("Capacity loss fraction")
plt.ylabel("Cluster")
save_plot(OUT_DIR / "top_clusters_capacity_loss.png")

plt.figure(figsize=(12, 5))
plt.hist(priority_table["delay_minutes_per_vehicle"], bins=30)
plt.title("Delay Minutes per Vehicle Distribution")
plt.xlabel("Delay minutes per vehicle")
plt.ylabel("Clusters")
save_plot(OUT_DIR / "delay_distribution.png")

Approved clustered rows: 92332
Clusters: 798


### It might take more than 2 hours so we will use DBSCAN or caching

In [ ]:
# =========================
# Phase 5 — Optimized Queueing, Capacity Loss, and Delay Estimation
# Fast version:
# - Vectorized cluster aggregation
# - Reuses Phase 4 dwell-time outputs
# - Uses cached / fallback road geometry by default
# - Optional OSMnx enrichment only if explicitly enabled
# =========================

import ast
import os
import warnings
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# -------------------------
# Config
# -------------------------
PHASE4_MERGED_PATH = Path("phase4_outputs_1/phase4_merged_with_prior_scores.csv")
PHASE4_CLUSTER_PATH = Path("phase4_outputs_1/phase4_cluster_mu_summary.csv")
PHASE5_ROAD_CACHE_PATH = Path("phase5_outputs_1/phase5_road_context_cache.csv")  # optional cache reuse

OUT_DIR = Path("phase5_outputs_1")
OUT_DIR.mkdir(parents=True, exist_ok=True)

EPS = 1e-9
TOP_N = 25

# Doc / paper constants
ALPHA_BPR = 0.56
BETA_BPR = 2.12

# Default geometry assumptions when OSMnx data is unavailable
DEFAULT_LANES = 2
DEFAULT_WIDTH_PER_LANE_M = 3.5
DEFAULT_CARRIAGEWAY_WIDTH_M = DEFAULT_LANES * DEFAULT_WIDTH_PER_LANE_M
DEFAULT_LINK_LENGTH_M = 250.0

# Set to True only if you really want OSMnx enrichment
ENABLE_OSMNX = os.environ.get("ENABLE_OSMNX_PHASE5", "1").strip() == "1"

ROAD_CLASS_SPEED_KMH = {
    "motorway": 60.0,
    "trunk": 55.0,
    "primary": 45.0,
    "secondary": 40.0,
    "tertiary": 35.0,
    "unclassified": 30.0,
    "residential": 30.0,
    "living_street": 20.0,
    "service": 25.0,
    "road": 30.0,
}

ROAD_CLASS_BASE_CAPACITY_PER_LANE = {
    "motorway": 2200.0,
    "trunk": 2100.0,
    "primary": 1900.0,
    "secondary": 1800.0,
    "tertiary": 1700.0,
    "unclassified": 1650.0,
    "residential": 1500.0,
    "living_street": 1200.0,
    "service": 1400.0,
    "road": 1600.0,
}

VEHICLE_WIDTH_M = {
    "SCOOTER": 0.80,
    "MOTOR CYCLE": 0.90,
    "MOTORCYCLE": 0.90,
    "BICYCLE": 0.60,
    "CYCLE": 0.60,
    "PASSENGER AUTO": 1.60,
    "AUTO": 1.60,
    "CAR": 1.90,
    "SUV": 2.00,
    "JEEP": 2.00,
    "VAN": 2.20,
    "TEMPO": 2.20,
    "BUS": 2.60,
    "TRUCK": 2.60,
    "LORRY": 2.60,
    "TANKER": 2.80,
    "TRACTOR": 2.20,
    "MINI TRUCK": 2.20,
    "AMBULANCE": 2.00,
    "UNKNOWN": 1.90,
}

# -------------------------
# Helpers
# -------------------------
def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def safe_ratio(a, b):
    return a / (b + EPS)

def minmax(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce").fillna(0.0).astype(float)
    if s.nunique(dropna=True) <= 1:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.min()) / (s.max() - s.min() + EPS)

def parse_listlike(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    s = str(value).strip()
    if not s:
        return []
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, (list, tuple)):
            return list(parsed)
        return [parsed]
    except Exception:
        if s.startswith("[") and s.endswith("]"):
            s = s[1:-1]
        parts = [p.strip().strip("'").strip('"') for p in s.split(",")]
        return [p for p in parts if p]

def parse_datetime_ist(df: pd.DataFrame) -> pd.Series:
    if "created_datetime_ist" in df.columns:
        ts = pd.to_datetime(df["created_datetime_ist"], errors="coerce", utc=True)
        if ts.notna().any():
            return ts.dt.tz_convert("Asia/Kolkata")
        ts = pd.to_datetime(df["created_datetime_ist"], errors="coerce")
        return ts.dt.tz_localize("Asia/Kolkata", nonexistent="NaT", ambiguous="NaT")

    if "created_datetime_parsed" in df.columns:
        ts = pd.to_datetime(df["created_datetime_parsed"], errors="coerce", utc=True)
        return ts.dt.tz_convert("Asia/Kolkata")

    if "created_datetime" not in df.columns:
        raise ValueError("Missing created_datetime / created_datetime_ist column.")

    ts = pd.to_datetime(df["created_datetime"], errors="coerce", utc=True)
    return ts.dt.tz_convert("Asia/Kolkata")

def week_start_monday(series):
    dt = pd.to_datetime(series, errors="coerce", utc=True).dt.tz_convert("Asia/Kolkata")
    week_start = dt.dt.normalize() - pd.to_timedelta(dt.dt.weekday, unit="D")
    return week_start.dt.tz_localize(None)

def make_hotspot_unit(row):
    junction = clean_text(row.get("junction_name", ""))
    if junction and junction.upper() != "NO JUNCTION":
        return f"JUNCTION::{junction}"
    station = clean_text(row.get("police_station", "UNKNOWN"))
    if not station:
        station = "UNKNOWN"
    return f"POLICE_STATION::{station}"

def standardize_cluster_col(df: pd.DataFrame) -> str:
    for c in ["st_dbscan_cluster_id", "cluster_id", "dbscan_cluster_id"]:
        if c in df.columns:
            return c
    raise ValueError("No cluster id column found.")

def standardize_vehicle_col(df: pd.DataFrame) -> str:
    if "canonical_vehicle_number" in df.columns:
        return "canonical_vehicle_number"
    if "vehicle_number" in df.columns:
        return "vehicle_number"
    raise ValueError("No vehicle number column found.")

def standardize_vehicle_type_col(df: pd.DataFrame) -> str:
    if "canonical_vehicle_type" in df.columns:
        return "canonical_vehicle_type"
    if "vehicle_type" in df.columns:
        return "vehicle_type"
    df["canonical_vehicle_type"] = "UNKNOWN"
    return "canonical_vehicle_type"

def road_speed_kmh(road_class):
    return ROAD_CLASS_SPEED_KMH.get(road_class, 30.0)

def base_capacity_per_lane(road_class):
    return ROAD_CLASS_BASE_CAPACITY_PER_LANE.get(road_class, 1600.0)

def vehicle_width_m(vehicle_type):
    vt = clean_text(vehicle_type).upper()
    return VEHICLE_WIDTH_M.get(vt, 1.90)

def normalize_road_class(value):
    if pd.isna(value):
        return "road"
    s = str(value).strip().lower()
    if not s:
        return "road"
    if isinstance(value, list):
        s = str(value[0]).strip().lower() if value else "road"

    # If OSM returns multiple tags or strings like "['primary', 'secondary']"
    if s.startswith("["):
        vals = parse_listlike(value)
        for v in vals:
            vv = clean_text(v).lower().replace(" ", "_")
            if vv in ROAD_CLASS_SPEED_KMH:
                return vv
        return "road"

    s = s.replace(" ", "_")
    mapping = {
        "motorway_link": "motorway",
        "trunk_link": "trunk",
        "primary_link": "primary",
        "secondary_link": "secondary",
        "tertiary_link": "tertiary",
    }
    s = mapping.get(s, s)
    return s if s in ROAD_CLASS_SPEED_KMH else "road"

def save_plot(path):
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()

# -------------------------
# Load Phase 4 outputs
# -------------------------
if not PHASE4_MERGED_PATH.exists():
    raise FileNotFoundError(
        "Phase 4 merged output not found. Expected phase4_outputs/phase4_merged_with_prior_scores.csv"
    )

df = pd.read_csv(PHASE4_MERGED_PATH, low_memory=False)
cluster_col = standardize_cluster_col(df)
vehicle_col = standardize_vehicle_col(df)
vehicle_type_col = standardize_vehicle_type_col(df)

df["created_datetime_ist"] = parse_datetime_ist(df)
df["created_datetime_ist_naive"] = df["created_datetime_ist"].dt.tz_localize(None)
df["week_start"] = week_start_monday(df["created_datetime"])

if "validation_status_clean" in df.columns:
    df["validation_status_clean"] = df["validation_status_clean"].fillna("").astype(str).str.lower()
elif "validation_status" in df.columns:
    df["validation_status_clean"] = df["validation_status"].fillna("").astype(str).str.lower()
else:
    df["validation_status_clean"] = "approved"

if "severity_score" not in df.columns:
    df["severity_score"] = 1.0

if "hotspot_unit" not in df.columns:
    if {"junction_name", "police_station"}.issubset(df.columns):
        df["hotspot_unit"] = df.apply(make_hotspot_unit, axis=1)
    else:
        df["hotspot_unit"] = "UNKNOWN"

approved = df.loc[df["validation_status_clean"].eq("approved")].copy()
approved = approved.dropna(subset=[cluster_col, "created_datetime_ist_naive", "latitude", "longitude"]).copy()
approved = approved.loc[approved[cluster_col].ne(-1)].copy()
approved = approved.loc[approved[vehicle_col].fillna("").astype(str).str.strip().ne("")].copy()
approved = approved.reset_index(drop=True)

if approved.empty:
    raise ValueError("No approved clustered rows available for Stage 5.")

print("Approved clustered rows:", len(approved))
print("Clusters:", approved[cluster_col].nunique())

# -------------------------
# Stage 5a — λ (arrival rate) by cluster
# Vectorized weekly/peak-window estimation
# -------------------------
approved["hour_ist"] = approved["created_datetime_ist"].dt.hour
approved["peak_window_flag"] = approved["hour_ist"].between(8, 12, inclusive="both").astype(int)

cluster_lambda = (
    approved.groupby(cluster_col)
    .agg(
        records_total=(vehicle_col, "size"),
        records_peak_window=("peak_window_flag", "sum"),
        records_peak_hour=(vehicle_col, lambda s: approved.loc[s.index, "hour_ist"].value_counts().max()),
        first_seen=("created_datetime_ist_naive", "min"),
        last_seen=("created_datetime_ist_naive", "max"),
        centroid_lat=("latitude", "mean"),
        centroid_lon=("longitude", "mean"),
        severity_sum=("severity_score", "sum"),
        severity_mean=("severity_score", "mean"),
        unique_vehicles=(vehicle_col, "nunique"),
        unique_vehicle_types=(vehicle_type_col, "nunique"),
    )
    .reset_index()
)

cluster_lambda["lambda_hr_peak_window"] = cluster_lambda["records_peak_window"] / 5.0
cluster_lambda["lambda_hr_peak_hour"] = pd.to_numeric(cluster_lambda["records_peak_hour"], errors="coerce").fillna(0.0)

# -------------------------
# Stage 4 carry-over — μ / dwell time
# Prefer Phase 4 summary; no recomputation here
# -------------------------
if not PHASE4_CLUSTER_PATH.exists():
    raise FileNotFoundError(
        "Phase 4 cluster summary not found. Expected phase4_outputs/phase4_cluster_mu_summary.csv"
    )

mu_df = pd.read_csv(PHASE4_CLUSTER_PATH, low_memory=False)
if cluster_col not in mu_df.columns and "st_dbscan_cluster_id" in mu_df.columns:
    mu_df = mu_df.rename(columns={"st_dbscan_cluster_id": cluster_col})

keep_cols = [cluster_col]
for c in [
    "cluster_label", "gap_count", "mean_dwell_minutes",
    "median_dwell_minutes", "std_dwell_minutes",
    "mu_departures_per_hour"
]:
    if c in mu_df.columns:
        keep_cols.append(c)

mu_df = mu_df[keep_cols].drop_duplicates(cluster_col)
cluster_table = cluster_lambda.merge(mu_df, on=cluster_col, how="left")

if "mu_departures_per_hour" not in cluster_table.columns:
    cluster_table["mu_departures_per_hour"] = 60.0 / (pd.to_numeric(cluster_table["mean_dwell_minutes"], errors="coerce") + EPS)

cluster_table["mean_dwell_minutes"] = pd.to_numeric(cluster_table["mean_dwell_minutes"], errors="coerce")
cluster_table["mu_departures_per_hour"] = pd.to_numeric(cluster_table["mu_departures_per_hour"], errors="coerce")
cluster_table["mu_departures_per_hour"] = cluster_table["mu_departures_per_hour"].fillna(
    60.0 / (cluster_table["mean_dwell_minutes"] + EPS)
)

cluster_table["blocking_vehicles_L"] = (
    cluster_table["lambda_hr_peak_window"] / (cluster_table["mu_departures_per_hour"] + EPS)
)

# -------------------------
# Vehicle mix / average blocking width
# -------------------------
vehicle_mix = (
    approved.groupby([cluster_col, vehicle_type_col])
    .size()
    .reset_index(name="count")
    .rename(columns={vehicle_type_col: "vehicle_type"})
)
vehicle_mix["vehicle_type"] = vehicle_mix["vehicle_type"].fillna("UNKNOWN").astype(str).str.strip()
vehicle_mix["vehicle_width_m"] = vehicle_mix["vehicle_type"].apply(vehicle_width_m)

cluster_width = (
    vehicle_mix.groupby(cluster_col)
    .apply(lambda g: safe_ratio((g["count"] * g["vehicle_width_m"]).sum(), g["count"].sum()))
    .reset_index(name="vehicle_width_avg_m")
)

cluster_table = cluster_table.merge(cluster_width, on=cluster_col, how="left")
cluster_table["vehicle_width_avg_m"] = cluster_table["vehicle_width_avg_m"].fillna(1.90)

# -------------------------
# Trend acceleration / growth multiplier
# -------------------------
weekly_counts = (
    approved.groupby([cluster_col, approved["created_datetime_ist"].dt.to_period("W-MON").apply(lambda p: p.start_time)])
    .size()
    .reset_index(name="weekly_count")
    .rename(columns={"created_datetime_ist": "cluster_week_start"})
    .sort_values([cluster_col, "cluster_week_start"])
)

growth_rows = []
for cid, g in weekly_counts.groupby(cluster_col, sort=False):
    counts = g["weekly_count"].to_numpy(dtype=float)
    if len(counts) < 2:
        first_mean = float(counts.mean()) if len(counts) else 0.0
        second_mean = first_mean
        growth_pct = 0.0
    else:
        mid = max(1, len(counts) // 2)
        first_half = counts[:mid]
        second_half = counts[mid:]
        first_mean = float(first_half.mean()) if len(first_half) else 0.0
        second_mean = float(second_half.mean()) if len(second_half) else 0.0
        growth_pct = safe_ratio((second_mean - first_mean), first_mean) if first_mean > 0 else second_mean

    growth_multiplier = max(0.5, 1.0 + max(0.0, growth_pct))
    growth_rows.append({
        cluster_col: cid,
        "growth_first_half_mean": first_mean,
        "growth_second_half_mean": second_mean,
        "growth_pct_change": growth_pct,
        "growth_multiplier": growth_multiplier,
    })

growth_df = pd.DataFrame(growth_rows)

# -------------------------
# Road-network context
# Optimized strategy:
# 1) Reuse cache if available
# 2) If OSMnx disabled, use defaults
# 3) If enabled, enrich only unique rounded centroids and cache results
# -------------------------
road_context_df = None

if PHASE5_ROAD_CACHE_PATH.exists():
    road_context_df = pd.read_csv(PHASE5_ROAD_CACHE_PATH, low_memory=False)
    if cluster_col not in road_context_df.columns and "st_dbscan_cluster_id" in road_context_df.columns:
        road_context_df = road_context_df.rename(columns={"st_dbscan_cluster_id": cluster_col})

if road_context_df is None:
    # Default fallback context
    road_context_df = cluster_table[[cluster_col, "centroid_lat", "centroid_lon"]].copy()
    road_context_df["geometry_source"] = "fallback"
    road_context_df["road_class"] = "road"
    road_context_df["lane_count"] = DEFAULT_LANES
    road_context_df["carriageway_width_m"] = DEFAULT_CARRIAGEWAY_WIDTH_M
    road_context_df["link_length_m"] = DEFAULT_LINK_LENGTH_M
    road_context_df["junction_degree"] = 4
    road_context_df["betweenness_centrality"] = 0.0

    if ENABLE_OSMNX:
        try:
            import osmnx as ox
            import networkx as nx

            @lru_cache(maxsize=512)
            def enrich_point(lat_round, lon_round):
                lat = float(lat_round)
                lon = float(lon_round)
                result = {
                    "geometry_source": "fallback",
                    "road_class": "road",
                    "lane_count": DEFAULT_LANES,
                    "carriageway_width_m": DEFAULT_CARRIAGEWAY_WIDTH_M,
                    "link_length_m": DEFAULT_LINK_LENGTH_M,
                    "junction_degree": 4,
                    "betweenness_centrality": 0.0,
                }
                try:
                    G = ox.graph_from_point((lat, lon), dist=500, network_type="drive", simplify=True, retain_all=False)
                    nearest_node = ox.distance.nearest_nodes(G, X=lon, Y=lat)
                    nearest_edge = ox.distance.nearest_edges(G, X=lon, Y=lat)
                    u, v, k = nearest_edge
                    edge_data = G.edges[u, v, k]

                    highway = edge_data.get("highway", "road")
                    road_class = normalize_road_class(highway)

                    lanes = edge_data.get("lanes", None)
                    if isinstance(lanes, list):
                        lanes = lanes[0] if lanes else None
                    try:
                        lanes = int(round(float(lanes))) if lanes is not None else DEFAULT_LANES
                    except Exception:
                        lanes = DEFAULT_LANES
                    lanes = max(1, lanes)

                    width = edge_data.get("width", None)
                    if isinstance(width, list):
                        width = width[0] if width else None
                    try:
                        width = float(str(width).replace("m", "").strip()) if width is not None else lanes * DEFAULT_WIDTH_PER_LANE_M
                    except Exception:
                        width = lanes * DEFAULT_WIDTH_PER_LANE_M

                    length_m = float(edge_data.get("length", DEFAULT_LINK_LENGTH_M))

                    UG = nx.Graph(G)
                    bc = nx.betweenness_centrality(UG, normalized=True, weight="length")
                    centrality = float(bc.get(nearest_node, 0.0))
                    degree = int(UG.degree[nearest_node])

                    result.update({
                        "geometry_source": "osmnx",
                        "road_class": road_class,
                        "lane_count": lanes,
                        "carriageway_width_m": width,
                        "link_length_m": length_m,
                        "junction_degree": degree,
                        "betweenness_centrality": centrality,
                    })
                except Exception:
                    pass
                return result

            # Use rounded centroids to collapse many nearly identical calls
            rounded = cluster_table[["centroid_lat", "centroid_lon"]].copy()
            rounded["lat_r"] = rounded["centroid_lat"].round(4)
            rounded["lon_r"] = rounded["centroid_lon"].round(4)

            unique_points = rounded[["lat_r", "lon_r"]].drop_duplicates()
            enrich_rows = []
            for _, r in unique_points.iterrows():
                enrich_rows.append({"lat_r": r["lat_r"], "lon_r": r["lon_r"], **enrich_point(r["lat_r"], r["lon_r"])})
            enrich_df = pd.DataFrame(enrich_rows)

            road_context_df = rounded.merge(enrich_df, on=["lat_r", "lon_r"], how="left")
            road_context_df = road_context_df.drop(columns=["lat_r", "lon_r"])

            road_context_df = cluster_table[[cluster_col, "centroid_lat", "centroid_lon"]].merge(
                road_context_df, on=["centroid_lat", "centroid_lon"], how="left"
            )

        except Exception:
            # If OSMnx fails, stay with defaults
            road_context_df = cluster_table[[cluster_col, "centroid_lat", "centroid_lon"]].copy()
            road_context_df["geometry_source"] = "fallback"
            road_context_df["road_class"] = "road"
            road_context_df["lane_count"] = DEFAULT_LANES
            road_context_df["carriageway_width_m"] = DEFAULT_CARRIAGEWAY_WIDTH_M
            road_context_df["link_length_m"] = DEFAULT_LINK_LENGTH_M
            road_context_df["junction_degree"] = 4
            road_context_df["betweenness_centrality"] = 0.0

# Merge road context
cluster_table = cluster_table.merge(
    road_context_df[[c for c in road_context_df.columns if c in [
        cluster_col, "geometry_source", "road_class", "lane_count",
        "carriageway_width_m", "link_length_m", "junction_degree",
        "betweenness_centrality"
    ]]],
    on=cluster_col,
    how="left"
)

# Fill geometry defaults
cluster_table["geometry_source"] = cluster_table.get("geometry_source", "fallback").fillna("fallback")
cluster_table["road_class"] = cluster_table.get("road_class", "road").fillna("road").astype(str)
cluster_table["lane_count"] = pd.to_numeric(cluster_table.get("lane_count", DEFAULT_LANES), errors="coerce").fillna(DEFAULT_LANES).astype(int)
cluster_table["carriageway_width_m"] = pd.to_numeric(cluster_table.get("carriageway_width_m", DEFAULT_CARRIAGEWAY_WIDTH_M), errors="coerce").fillna(DEFAULT_CARRIAGEWAY_WIDTH_M)
cluster_table["link_length_m"] = pd.to_numeric(cluster_table.get("link_length_m", DEFAULT_LINK_LENGTH_M), errors="coerce").fillna(DEFAULT_LINK_LENGTH_M)
cluster_table["junction_degree"] = pd.to_numeric(cluster_table.get("junction_degree", 4), errors="coerce").fillna(4.0)
cluster_table["betweenness_centrality"] = pd.to_numeric(cluster_table.get("betweenness_centrality", 0.0), errors="coerce").fillna(0.0)

cluster_table["junction_degree_norm"] = minmax(cluster_table["junction_degree"])
cluster_table["betweenness_centrality_norm"] = minmax(cluster_table["betweenness_centrality"])
cluster_table["criticality_factor"] = 1.0 + 0.5 * cluster_table["junction_degree_norm"] + 0.5 * cluster_table["betweenness_centrality_norm"]

# -------------------------
# Stage 5b — Zhao capacity loss
# -------------------------
cluster_table["base_saturation_per_lane_pcu_hr"] = cluster_table["road_class"].map(base_capacity_per_lane).fillna(1600.0)
cluster_table["free_flow_speed_kmh"] = cluster_table["road_class"].apply(road_speed_kmh)
cluster_table["free_flow_time_min"] = cluster_table["link_length_m"] * 60.0 / (cluster_table["free_flow_speed_kmh"] * 1000.0 + EPS)

cluster_table["width_factor"] = cluster_table["carriageway_width_m"] / (cluster_table["lane_count"] * DEFAULT_WIDTH_PER_LANE_M + EPS)
cluster_table["width_factor"] = cluster_table["width_factor"].clip(0.70, 1.20)

cluster_table["base_capacity_pcu_hr"] = (
    cluster_table["base_saturation_per_lane_pcu_hr"] *
    cluster_table["lane_count"] *
    cluster_table["width_factor"]
)

cluster_table["blocked_width_fraction_raw"] = (
    cluster_table["blocking_vehicles_L"] * cluster_table["vehicle_width_avg_m"]
) / (cluster_table["carriageway_width_m"] + EPS)

cluster_table["blocked_width_fraction"] = cluster_table["blocked_width_fraction_raw"].clip(0.0, 0.95)
cluster_table["reduced_capacity_pcu_hr"] = (
    cluster_table["base_capacity_pcu_hr"] * (1.0 - cluster_table["blocked_width_fraction"])
).clip(lower=cluster_table["base_capacity_pcu_hr"] * 0.10)

cluster_table["capacity_loss_pct"] = 1.0 - safe_ratio(cluster_table["reduced_capacity_pcu_hr"], cluster_table["base_capacity_pcu_hr"])

# -------------------------
# Stage 5c — Modified BPR delay
# -------------------------
cluster_table["V_over_C0"] = cluster_table["lambda_hr_peak_window"] / (cluster_table["base_capacity_pcu_hr"] + EPS)
cluster_table["V_over_Cp"] = cluster_table["lambda_hr_peak_window"] / (cluster_table["reduced_capacity_pcu_hr"] + EPS)

cluster_table["delay_minutes_per_vehicle"] = (
    cluster_table["free_flow_time_min"] * ALPHA_BPR *
    (np.power(cluster_table["V_over_Cp"], BETA_BPR) - np.power(cluster_table["V_over_C0"], BETA_BPR))
).clip(lower=0.0)

# -------------------------
# Growth / repeat offender support for Stage 6
# -------------------------
cluster_table = cluster_table.merge(growth_df, on=cluster_col, how="left")
cluster_table["growth_pct_change"] = pd.to_numeric(cluster_table["growth_pct_change"], errors="coerce").fillna(0.0)
cluster_table["growth_multiplier"] = pd.to_numeric(cluster_table["growth_multiplier"], errors="coerce").fillna(1.0)

repeat_vehicle_counts = (
    approved.groupby([cluster_col, vehicle_col])
    .size()
    .reset_index(name="vehicle_cluster_count")
)
repeat_vehicle_counts["repeat_flag_2plus"] = (repeat_vehicle_counts["vehicle_cluster_count"] >= 2).astype(int)
repeat_vehicle_counts["chronic_flag_5plus"] = (repeat_vehicle_counts["vehicle_cluster_count"] >= 5).astype(int)

cluster_repeat_summary = (
    repeat_vehicle_counts.groupby(cluster_col)
    .agg(
        repeat_vehicle_count_2plus=("repeat_flag_2plus", "sum"),
        chronic_vehicle_count_5plus=("chronic_flag_5plus", "sum"),
    )
    .reset_index()
)
cluster_table = cluster_table.merge(cluster_repeat_summary, on=cluster_col, how="left")
cluster_table["repeat_vehicle_count_2plus"] = cluster_table["repeat_vehicle_count_2plus"].fillna(0).astype(int)
cluster_table["chronic_vehicle_count_5plus"] = cluster_table["chronic_vehicle_count_5plus"].fillna(0).astype(int)

# -------------------------
# Final Stage 5 tables
# -------------------------
priority_table = cluster_table.sort_values(
    ["delay_minutes_per_vehicle", "lambda_hr_peak_window", "blocking_vehicles_L"],
    ascending=[False, False, False]
).reset_index(drop=True)

priority_table["stage5_rank"] = np.arange(1, len(priority_table) + 1)
priority_table["delay_minutes_norm"] = minmax(priority_table["delay_minutes_per_vehicle"])
priority_table["lambda_norm"] = minmax(priority_table["lambda_hr_peak_window"])
priority_table["severity_sum_norm"] = minmax(priority_table["severity_sum"])

# Stage 5 handoff view
handoff_cols = [
    "stage5_rank", cluster_col, "cluster_label", "records_total", "records_peak_window",
    "lambda_hr_peak_window", "mean_dwell_minutes", "mu_departures_per_hour", "blocking_vehicles_L",
    "road_class", "lane_count", "carriageway_width_m", "link_length_m", "base_capacity_pcu_hr",
    "reduced_capacity_pcu_hr", "capacity_loss_pct", "free_flow_time_min", "delay_minutes_per_vehicle",
    "junction_degree", "betweenness_centrality", "criticality_factor", "growth_pct_change",
    "growth_multiplier", "vehicle_width_avg_m", "dominant_vehicle_type", "unique_vehicles",
    "unique_vehicle_types", "repeat_vehicle_count_2plus", "chronic_vehicle_count_5plus",
    "geometry_source", "centroid_lat", "centroid_lon"
]
handoff_cols = [c for c in handoff_cols if c in priority_table.columns]

# Record-level view for Stage 6
stage5_record_view = approved.merge(
    priority_table[[
        cluster_col, "cluster_label", "records_total", "lambda_hr_peak_window", "mean_dwell_minutes",
        "mu_departures_per_hour", "blocking_vehicles_L", "road_class", "lane_count",
        "carriageway_width_m", "link_length_m", "base_capacity_pcu_hr", "reduced_capacity_pcu_hr",
        "capacity_loss_pct", "free_flow_time_min", "delay_minutes_per_vehicle",
        "junction_degree", "betweenness_centrality", "criticality_factor",
        "growth_pct_change", "growth_multiplier", "stage5_rank"
    ]],
    on=cluster_col,
    how="left"
)

# Chronic offenders table
vehicle_offenders = (
    approved.groupby(vehicle_col)
    .agg(
        total_violations=(vehicle_col, "size"),
        unique_clusters=(cluster_col, "nunique"),
        unique_hotspots=("hotspot_unit", "nunique"),
        first_seen=("created_datetime_ist_naive", "min"),
        last_seen=("created_datetime_ist_naive", "max"),
        dominant_vehicle_type=(vehicle_type_col, lambda s: s.mode().iloc[0] if not s.mode().empty else "UNKNOWN"),
    )
    .reset_index()
    .rename(columns={vehicle_col: "vehicle_number"})
    .sort_values(["total_violations", "unique_clusters"], ascending=[False, False])
)
vehicle_offenders["chronic_offender_flag"] = (vehicle_offenders["total_violations"] >= 5).astype(int)
chronic_offenders = vehicle_offenders.loc[vehicle_offenders["chronic_offender_flag"].eq(1)].copy()

# -------------------------
# Save outputs
# -------------------------
cluster_table.to_csv(OUT_DIR / "phase5_cluster_capacity_loss.csv", index=False)
priority_table[handoff_cols].to_csv(OUT_DIR / "phase5_priority_table.csv", index=False)
priority_table.to_csv(OUT_DIR / "phase5_priority_table_full.csv", index=False)
stage5_record_view.to_csv(OUT_DIR / "phase5_enriched_records.csv", index=False)
vehicle_mix.to_csv(OUT_DIR / "phase5_vehicle_mix.csv", index=False)
weekly_counts.to_csv(OUT_DIR / "phase5_weekly_cluster_counts.csv", index=False)
growth_df.to_csv(OUT_DIR / "phase5_growth_summary.csv", index=False)
chronic_offenders.to_csv(OUT_DIR / "phase5_chronic_offenders.csv", index=False)
road_context_df.to_csv(OUT_DIR / "phase5_road_context_cache.csv", index=False)

priority_table[[
    "stage5_rank", cluster_col, "cluster_label", "delay_minutes_per_vehicle",
    "lambda_hr_peak_window", "mu_departures_per_hour", "blocking_vehicles_L",
    "capacity_loss_pct", "criticality_factor", "growth_multiplier", "road_class",
    "lane_count", "carriageway_width_m", "free_flow_time_min"
]].to_csv(OUT_DIR / "phase5_stage5_handoff.csv", index=False)

# -------------------------
# Console summary
# -------------------------
print("Phase 5 complete")
print("Approved clustered rows:", len(approved))
print("Clusters scored:", len(priority_table))
print("Chronic offenders:", len(chronic_offenders))
print("OSMnx enabled:", ENABLE_OSMNX)
print("Outputs saved to:", OUT_DIR.resolve())

print("\nTop 10 clusters by delay:")
print(
    priority_table[[
        "stage5_rank", cluster_col, "cluster_label", "delay_minutes_per_vehicle",
        "lambda_hr_peak_window", "mu_departures_per_hour", "blocking_vehicles_L",
        "capacity_loss_pct", "criticality_factor", "growth_multiplier"
    ]].head(10).to_string(index=False)
)

# -------------------------
# Plots
# -------------------------
plt.figure(figsize=(12, 7))
top_delay = priority_table.head(TOP_N).sort_values("delay_minutes_per_vehicle", ascending=True)
plt.barh(top_delay["cluster_label"] if "cluster_label" in top_delay.columns else top_delay[cluster_col], top_delay["delay_minutes_per_vehicle"])
plt.title("Top Clusters by Delay Minutes per Vehicle")
plt.xlabel("Delay minutes per vehicle")
plt.ylabel("Cluster")
save_plot(OUT_DIR / "top_clusters_delay.png")

plt.figure(figsize=(12, 7))
top_capacity = priority_table.head(TOP_N).sort_values("capacity_loss_pct", ascending=True)
plt.barh(top_capacity["cluster_label"] if "cluster_label" in top_capacity.columns else top_capacity[cluster_col], top_capacity["capacity_loss_pct"])
plt.title("Top Clusters by Capacity Loss")
plt.xlabel("Capacity loss fraction")
plt.ylabel("Cluster")
save_plot(OUT_DIR / "top_clusters_capacity_loss.png")

plt.figure(figsize=(12, 5))
plt.hist(priority_table["delay_minutes_per_vehicle"], bins=30)
plt.title("Delay Minutes per Vehicle Distribution")
plt.xlabel("Delay minutes per vehicle")
plt.ylabel("Clusters")
save_plot(OUT_DIR / "delay_distribution.png")

Approved clustered rows: 92332
Clusters: 798
Phase 5 complete
Approved clustered rows: 92332
Clusters scored: 798
Chronic offenders: 638
OSMnx enabled: True
Outputs saved to: /content/phase5_outputs_1

Top 10 clusters by delay:
 stage5_rank  st_dbscan_cluster_id                                cluster_label  delay_minutes_per_vehicle  lambda_hr_peak_window  mu_departures_per_hour  blocking_vehicles_L  capacity_loss_pct  criticality_factor  growth_multiplier
           1                     2            JUNCTION::BTP040 - Elite Junction                  13.737407                 2014.8                0.592115          3402.719514                0.9                 1.0           1.000000
           2                     3     JUNCTION::BTP051 - Safina Plaza Junction                   4.902220                 1239.2                0.661847          1872.337027                0.9                 1.0           1.000000
           3                    19                 POLICE_STATION::Malles

In [ ]:
# =========================
# Phase 6 — Congestion Cost Score (CCS) Computation
# Corrected version:
# - Fills missing labels and numeric fields
# - Uses smoothed normalization to avoid zero-collapse
# - Produces final CCS ranking, dispatch table, alerts, chronic list
# =========================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# -------------------------
# Config
# -------------------------
PHASE5_CLUSTER_PATH = Path("phase5_outputs_1/phase5_cluster_capacity_loss.csv")
PHASE5_PRIORITY_PATH = Path("phase5_outputs_1/phase5_priority_table_full.csv")
PHASE5_GROWTH_PATH = Path("phase5_outputs_1/phase5_growth_summary.csv")
PHASE5_CHRONIC_PATH = Path("phase5_outputs_1/phase5_chronic_offenders.csv")
PHASE5_ENRICHED_RECORDS_PATH = Path("phase5_outputs_1/phase5_enriched_records.csv")

PHASE4_MERGED_PATH = Path("phase4_outputs_1/phase4_merged_with_prior_scores.csv")
PHASE3_SUMMARY_PATH = Path("phase3_outputs_1/phase3_cluster_summary.csv")

OUT_DIR = Path("phase6_outputs_1")
OUT_DIR.mkdir(parents=True, exist_ok=True)

EPS = 1e-9
TOP_N = 25

# -------------------------
# Helpers
# -------------------------
def minmax(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce").fillna(0.0).astype(float)
    if s.nunique(dropna=True) <= 1:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.min()) / (s.max() - s.min() + EPS)

def smooth_norm(s: pd.Series, floor: float = 0.10) -> pd.Series:
    """
    Prevents collapse to zero when a feature is nearly constant.
    Returns values in [floor, 1.0].
    """
    return floor + (1.0 - floor) * minmax(s)

def load_first_existing(paths):
    for p in paths:
        if p.exists():
            return pd.read_csv(p, low_memory=False), p
    return None, None

def standardize_cluster_col(df: pd.DataFrame) -> str:
    for c in ["st_dbscan_cluster_id", "cluster_id", "dbscan_cluster_id"]:
        if c in df.columns:
            return c
    raise ValueError("No cluster id column found.")

def standardize_vehicle_col(df: pd.DataFrame) -> str:
    if "canonical_vehicle_number" in df.columns:
        return "canonical_vehicle_number"
    if "vehicle_number" in df.columns:
        return "vehicle_number"
    raise ValueError("No vehicle number column found.")

def standardize_vehicle_type_col(df: pd.DataFrame):
    if "canonical_vehicle_type" in df.columns:
        return "canonical_vehicle_type"
    if "vehicle_type" in df.columns:
        return "vehicle_type"
    return None

def derive_cluster_label(df: pd.DataFrame, cluster_col: str) -> pd.Series:
    """
    Backfill cluster labels from cluster_label -> hotspot_unit -> junction_name/police_station -> cluster id.
    """
    label = pd.Series([""] * len(df), index=df.index, dtype="object")

    if "cluster_label" in df.columns:
        label = df["cluster_label"].fillna("").astype(str).str.strip()

    if "hotspot_unit" in df.columns:
        empty = label.eq("")
        label.loc[empty] = df.loc[empty, "hotspot_unit"].fillna("").astype(str).str.strip()

    # Try to improve labels from junction_name / police_station if hotspot_unit is missing or generic
    if {"junction_name", "police_station"}.issubset(df.columns):
        empty = label.eq("") | label.str.lower().eq("nan")
        junction = df.loc[empty, "junction_name"].fillna("").astype(str).str.strip()
        ps = df.loc[empty, "police_station"].fillna("").astype(str).str.strip()

        junction_ok = junction.ne("") & junction.str.upper().ne("NO JUNCTION")
        label.loc[empty] = np.where(
            junction_ok,
            "JUNCTION::" + junction.where(junction_ok, ""),
            "POLICE_STATION::" + ps.where(ps.ne(""), "UNKNOWN"),
        )

    empty = label.eq("") | label.str.lower().eq("nan")
    label.loc[empty] = "CLUSTER::" + df.loc[empty, cluster_col].astype(str)

    return label

def road_speed_kmh(road_class):
    ROAD_CLASS_SPEED_KMH = {
        "motorway": 60.0,
        "trunk": 55.0,
        "primary": 45.0,
        "secondary": 40.0,
        "tertiary": 35.0,
        "unclassified": 30.0,
        "residential": 30.0,
        "living_street": 20.0,
        "service": 25.0,
        "road": 30.0,
    }
    return ROAD_CLASS_SPEED_KMH.get(str(road_class).strip().lower(), 30.0)

def base_capacity_per_lane(road_class):
    ROAD_CLASS_BASE_CAPACITY_PER_LANE = {
        "motorway": 2200.0,
        "trunk": 2100.0,
        "primary": 1900.0,
        "secondary": 1800.0,
        "tertiary": 1700.0,
        "unclassified": 1650.0,
        "residential": 1500.0,
        "living_street": 1200.0,
        "service": 1400.0,
        "road": 1600.0,
    }
    return ROAD_CLASS_BASE_CAPACITY_PER_LANE.get(str(road_class).strip().lower(), 1600.0)

def vehicle_width_m(vehicle_type):
    VEHICLE_WIDTH_M = {
        "SCOOTER": 0.80,
        "MOTOR CYCLE": 0.90,
        "MOTORCYCLE": 0.90,
        "BICYCLE": 0.60,
        "CYCLE": 0.60,
        "PASSENGER AUTO": 1.60,
        "AUTO": 1.60,
        "CAR": 1.90,
        "SUV": 2.00,
        "JEEP": 2.00,
        "VAN": 2.20,
        "TEMPO": 2.20,
        "BUS": 2.60,
        "TRUCK": 2.60,
        "LORRY": 2.60,
        "TANKER": 2.80,
        "TRACTOR": 2.20,
        "MINI TRUCK": 2.20,
        "AMBULANCE": 2.00,
        "UNKNOWN": 1.90,
    }
    return VEHICLE_WIDTH_M.get(str(vehicle_type).strip().upper(), 1.90)

def save_plot(path):
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()

# -------------------------
# Load Phase 5 output
# -------------------------
cluster_df, cluster_src = load_first_existing([PHASE5_CLUSTER_PATH, PHASE5_PRIORITY_PATH])
if cluster_df is None:
    raise FileNotFoundError("Missing Phase 5 output. Expected phase5_outputs/*.csv")

cluster_col = standardize_cluster_col(cluster_df)

# -------------------------
# Backfill labels and key fields early
# -------------------------
cluster_df["cluster_label"] = derive_cluster_label(cluster_df, cluster_col)

# Fill important text cols
for col, default in [
    ("road_class", "road"),
    ("geometry_source", "fallback"),
    ("dominant_vehicle_type", "UNKNOWN"),
]:
    if col not in cluster_df.columns:
        cluster_df[col] = default
    cluster_df[col] = cluster_df[col].fillna(default).astype(str).replace({"nan": default, "": default})

# Fill numeric cols safely
numeric_defaults = {
    "delay_minutes_per_vehicle": 0.0,
    "lambda_hr_peak_window": 0.0,
    "lambda_hr_peak_hour": 0.0,
    "mu_departures_per_hour": 0.0,
    "mean_dwell_minutes": 0.0,
    "severity_sum": 0.0,
    "severity_mean": 0.0,
    "growth_pct_change": 0.0,
    "growth_multiplier": 1.0,
    "criticality_factor": 1.0,
    "capacity_loss_pct": 0.0,
    "blocking_vehicles_L": 0.0,
    "records_total": 0.0,
    "records_peak_window": 0.0,
    "records_peak_hour": 0.0,
    "junction_degree": 4.0,
    "betweenness_centrality": 0.0,
    "lane_count": 2.0,
    "carriageway_width_m": 7.0,
    "link_length_m": 250.0,
    "unique_vehicles": 0.0,
    "unique_vehicle_types": 0.0,
    "vehicle_width_avg_m": 1.9,
    "repeat_vehicle_count_2plus": 0.0,
    "chronic_vehicle_count_5plus": 0.0,
}

for col, default in numeric_defaults.items():
    if col not in cluster_df.columns:
        cluster_df[col] = default
    cluster_df[col] = pd.to_numeric(cluster_df[col], errors="coerce").fillna(default)

# Prefer the more informative lambda column
if "lambda_hr_peak_window" in cluster_df.columns and cluster_df["lambda_hr_peak_window"].abs().sum() > 0:
    lambda_col = "lambda_hr_peak_window"
elif "lambda_hr_peak_hour" in cluster_df.columns:
    lambda_col = "lambda_hr_peak_hour"
else:
    raise ValueError("No lambda column found in Phase 5 output.")

# If phase 5 priority file has growth / criticality already, keep them; otherwise backfill
if "growth_multiplier" not in cluster_df.columns:
    cluster_df["growth_multiplier"] = 1.0
if "growth_pct_change" not in cluster_df.columns:
    cluster_df["growth_pct_change"] = 0.0
if "criticality_factor" not in cluster_df.columns:
    cluster_df["criticality_factor"] = 1.0

# If a growth summary exists, merge it in
growth_df, _ = load_first_existing([PHASE5_GROWTH_PATH])
if growth_df is not None:
    if cluster_col not in growth_df.columns and "st_dbscan_cluster_id" in growth_df.columns:
        growth_df = growth_df.rename(columns={"st_dbscan_cluster_id": cluster_col})
    growth_keep = [cluster_col]
    for c in ["growth_first_half_mean", "growth_second_half_mean", "growth_pct_change", "growth_multiplier"]:
        if c in growth_df.columns:
            growth_keep.append(c)
    growth_df = growth_df[growth_keep].drop_duplicates(cluster_col)
    cluster_df = cluster_df.merge(growth_df, on=cluster_col, how="left", suffixes=("", "_g"))
    for c in ["growth_pct_change", "growth_multiplier"]:
        if f"{c}_g" in cluster_df.columns:
            cluster_df[c] = cluster_df[f"{c}_g"].combine_first(cluster_df[c])
            cluster_df.drop(columns=[f"{c}_g"], inplace=True)

# Refill after merge
for col, default in [("growth_pct_change", 0.0), ("growth_multiplier", 1.0)]:
    cluster_df[col] = pd.to_numeric(cluster_df[col], errors="coerce").fillna(default)

# Merge chronic counts if available
if "repeat_vehicle_count_2plus" not in cluster_df.columns:
    cluster_df["repeat_vehicle_count_2plus"] = 0
if "chronic_vehicle_count_5plus" not in cluster_df.columns:
    cluster_df["chronic_vehicle_count_5plus"] = 0

# Backfill cluster label again after merges
cluster_df["cluster_label"] = derive_cluster_label(cluster_df, cluster_col)

# -------------------------
# Ensure CCS inputs exist
# -------------------------
required_inputs = ["delay_minutes_per_vehicle", lambda_col, "severity_sum", "growth_multiplier", "criticality_factor"]
for c in required_inputs:
    if c not in cluster_df.columns:
        raise ValueError(f"Missing required CCS input: {c}")

cluster_df["delay_minutes_per_vehicle"] = pd.to_numeric(cluster_df["delay_minutes_per_vehicle"], errors="coerce").fillna(0.0).clip(lower=0.0)
cluster_df[lambda_col] = pd.to_numeric(cluster_df[lambda_col], errors="coerce").fillna(0.0).clip(lower=0.0)
cluster_df["severity_sum"] = pd.to_numeric(cluster_df["severity_sum"], errors="coerce").fillna(0.0).clip(lower=0.0)
cluster_df["growth_multiplier"] = pd.to_numeric(cluster_df["growth_multiplier"], errors="coerce").fillna(1.0).clip(lower=0.1)
cluster_df["criticality_factor"] = pd.to_numeric(cluster_df["criticality_factor"], errors="coerce").fillna(1.0).clip(lower=0.1)
cluster_df["growth_pct_change"] = pd.to_numeric(cluster_df["growth_pct_change"], errors="coerce").fillna(0.0)

# -------------------------
# Smoothed normalization for CCS
# -------------------------
cluster_df["delay_norm"] = smooth_norm(cluster_df["delay_minutes_per_vehicle"], floor=0.10)
cluster_df["lambda_norm"] = smooth_norm(cluster_df[lambda_col], floor=0.10)
cluster_df["severity_norm"] = smooth_norm(cluster_df["severity_sum"], floor=0.10)
cluster_df["growth_norm"] = smooth_norm(cluster_df["growth_multiplier"], floor=0.10)
cluster_df["criticality_norm"] = smooth_norm(cluster_df["criticality_factor"], floor=0.10)

# Raw product retained for traceability
cluster_df["ccs_raw_product"] = (
    cluster_df["delay_norm"] *
    cluster_df["lambda_norm"] *
    cluster_df["severity_norm"] *
    cluster_df["growth_norm"] *
    cluster_df["criticality_norm"]
)

# Geometric-mean style score keeps the ranking but avoids collapse to near-zero
cluster_df["ccs_score"] = 100.0 * np.power(cluster_df["ccs_raw_product"], 1.0 / 5.0)
cluster_df["ccs_score"] = cluster_df["ccs_score"].fillna(0.0)

# An additional readable linear blend for auditing
cluster_df["ccs_weighted_score"] = 100.0 * (
    0.25 * cluster_df["delay_norm"] +
    0.20 * cluster_df["lambda_norm"] +
    0.20 * cluster_df["severity_norm"] +
    0.15 * cluster_df["growth_norm"] +
    0.20 * cluster_df["criticality_norm"]
)

# Final ranking
rank_view = cluster_df.sort_values(
    ["ccs_score", "delay_minutes_per_vehicle", lambda_col],
    ascending=[False, False, False]
).reset_index(drop=True)
rank_view["ccs_rank"] = np.arange(1, len(rank_view) + 1)

# Risk banding
q80 = rank_view["ccs_score"].quantile(0.80) if len(rank_view) else 0
q60 = rank_view["ccs_score"].quantile(0.60) if len(rank_view) else 0
q40 = rank_view["ccs_score"].quantile(0.40) if len(rank_view) else 0

def risk_band(x):
    if x >= q80:
        return "Critical"
    if x >= q60:
        return "High"
    if x >= q40:
        return "Moderate"
    return "Watch"

rank_view["risk_band"] = rank_view["ccs_score"].apply(risk_band)

# Fix any remaining blank labels before export
rank_view["cluster_label"] = derive_cluster_label(rank_view, cluster_col)

# -------------------------
# Weekly dispatch priority table
# -------------------------
dispatch_cols = [
    "ccs_rank", cluster_col, "cluster_label", "risk_band",
    "ccs_score", "ccs_weighted_score",
    "delay_minutes_per_vehicle", lambda_col,
    "severity_sum", "severity_mean",
    "growth_pct_change", "growth_multiplier",
    "criticality_factor",
    "capacity_loss_pct", "blocking_vehicles_L",
    "road_class", "lane_count", "carriageway_width_m",
    "dominant_vehicle_type", "records_total", "unique_vehicles",
    "geometry_source",
]
dispatch_cols = [c for c in dispatch_cols if c in rank_view.columns]

weekly_dispatch = rank_view[dispatch_cols].copy()
weekly_dispatch.to_csv(OUT_DIR / "phase6_weekly_dispatch_priority_table.csv", index=False)

# -------------------------
# Hotspot CCS ranking
# -------------------------
hotspot_ranking = rank_view[[
    c for c in [
        "ccs_rank", cluster_col, "cluster_label", "risk_band",
        "ccs_score", "ccs_weighted_score", "delay_minutes_per_vehicle",
        lambda_col, "severity_sum", "growth_pct_change",
        "growth_multiplier", "criticality_factor",
        "capacity_loss_pct", "blocking_vehicles_L",
        "road_class", "lane_count", "carriageway_width_m",
        "dominant_vehicle_type", "records_total", "unique_vehicles",
        "geometry_source"
    ] if c in rank_view.columns
]].copy()

hotspot_ranking.to_csv(OUT_DIR / "phase6_hotspot_ranking_ccs.csv", index=False)

# -------------------------
# Emerging hotspot alerts
# -------------------------
alerts = rank_view.copy()
alerts["is_emerging"] = (
    (alerts["growth_pct_change"] > 0) &
    (alerts["records_total"] >= max(10, int(alerts["records_total"].median()))) &
    (alerts["growth_multiplier"] >= 1.0)
)

emerging_alerts = alerts.loc[alerts["is_emerging"]].copy()
if emerging_alerts.empty:
    emerging_alerts = alerts.sort_values(
        ["growth_pct_change", "growth_multiplier", "records_total"],
        ascending=[False, False, False]
    ).head(TOP_N).copy()
else:
    emerging_alerts = emerging_alerts.sort_values(
        ["growth_pct_change", "growth_multiplier", "records_total"],
        ascending=[False, False, False]
    ).head(TOP_N).copy()

if len(emerging_alerts):
    emerging_alerts["alert_level"] = np.where(
        emerging_alerts["growth_pct_change"] >= emerging_alerts["growth_pct_change"].quantile(0.80),
        "Emerging-Critical",
        np.where(
            emerging_alerts["growth_pct_change"] >= emerging_alerts["growth_pct_change"].quantile(0.60),
            "Emerging-High",
            "Emerging-Watch"
        )
    )
else:
    emerging_alerts["alert_level"] = []

emerging_cols = [
    "alert_level", "ccs_rank", cluster_col, "cluster_label",
    "growth_pct_change", "growth_multiplier",
    "ccs_score", "delay_minutes_per_vehicle", lambda_col,
    "severity_sum", "criticality_factor", "records_total",
    "dominant_vehicle_type", "risk_band"
]
emerging_cols = [c for c in emerging_cols if c in emerging_alerts.columns]
emerging_alerts[emerging_cols].to_csv(OUT_DIR / "phase6_emerging_hotspot_alerts.csv", index=False)

# -------------------------
# Chronic offender list
# -------------------------
if PHASE5_CHRONIC_PATH.exists():
    chronic_offenders = pd.read_csv(PHASE5_CHRONIC_PATH, low_memory=False)
else:
    if PHASE5_ENRICHED_RECORDS_PATH.exists():
        recs = pd.read_csv(PHASE5_ENRICHED_RECORDS_PATH, low_memory=False)
    elif PHASE4_MERGED_PATH.exists():
        recs = pd.read_csv(PHASE4_MERGED_PATH, low_memory=False)
    else:
        recs = None

    if recs is not None:
        rec_cluster_col = standardize_cluster_col(recs) if any(c in recs.columns for c in ["st_dbscan_cluster_id", "cluster_id", "dbscan_cluster_id"]) else None
        rec_vehicle_col = "canonical_vehicle_number" if "canonical_vehicle_number" in recs.columns else ("vehicle_number" if "vehicle_number" in recs.columns else None)

        if rec_cluster_col and rec_vehicle_col:
            chronic_offenders = (
                recs.groupby(rec_vehicle_col)
                .agg(
                    total_violations=(rec_vehicle_col, "size"),
                    unique_clusters=(rec_cluster_col, "nunique"),
                )
                .reset_index()
                .rename(columns={rec_vehicle_col: "vehicle_number"})
                .sort_values(["total_violations", "unique_clusters"], ascending=[False, False])
            )
            chronic_offenders["chronic_offender_flag"] = (chronic_offenders["total_violations"] >= 5).astype(int)
            chronic_offenders = chronic_offenders.loc[chronic_offenders["chronic_offender_flag"].eq(1)].copy()
        else:
            chronic_offenders = pd.DataFrame(columns=["vehicle_number", "total_violations", "chronic_offender_flag"])
    else:
        chronic_offenders = pd.DataFrame(columns=["vehicle_number", "total_violations", "chronic_offender_flag"])

chronic_offenders.to_csv(OUT_DIR / "phase6_chronic_offender_list.csv", index=False)

# -------------------------
# Enforcement recommendations
# -------------------------
recommendations = rank_view.copy()
recommendations["recommended_action"] = np.select(
    [
        recommendations["risk_band"].eq("Critical"),
        recommendations["risk_band"].eq("High"),
        recommendations["risk_band"].eq("Moderate"),
    ],
    [
        "Immediate patrol deployment",
        "Targeted enforcement + towing readiness",
        "Monitor and schedule peak-window checks",
    ],
    default="Routine monitoring",
)

recommendations["priority_message"] = (
    recommendations["cluster_label"].astype(str) + " | " +
    recommendations["risk_band"].astype(str) + " | CCS=" +
    recommendations["ccs_score"].round(3).astype(str)
)

recommendations_cols = [
    "ccs_rank", cluster_col, "cluster_label", "risk_band", "recommended_action",
    "delay_minutes_per_vehicle", lambda_col, "severity_sum",
    "growth_pct_change", "criticality_factor", "ccs_score",
    "dominant_vehicle_type", "road_class", "lane_count", "geometry_source"
]
recommendations_cols = [c for c in recommendations_cols if c in recommendations.columns]
recommendations[recommendations_cols].to_csv(OUT_DIR / "phase6_enforcement_recommendations.csv", index=False)

# -------------------------
# Full scored output and handoff
# -------------------------
rank_view.to_csv(OUT_DIR / "phase6_cluster_ccs_full.csv", index=False)

handoff_cols = [
    "ccs_rank", cluster_col, "cluster_label", "risk_band",
    "ccs_score", "delay_minutes_per_vehicle", lambda_col,
    "severity_sum", "growth_pct_change", "growth_multiplier",
    "criticality_factor", "capacity_loss_pct", "blocking_vehicles_L",
    "road_class", "lane_count", "carriageway_width_m",
    "dominant_vehicle_type", "records_total", "unique_vehicles",
    "geometry_source"
]
handoff_cols = [c for c in handoff_cols if c in rank_view.columns]
rank_view[handoff_cols].to_csv(OUT_DIR / "phase6_stage6_handoff.csv", index=False)

# Also useful for auditing
cluster_df.to_csv(OUT_DIR / "phase6_cluster_ccs_audit_full.csv", index=False)

# -------------------------
# Console summary
# -------------------------
print("Phase 6 complete")
print("Input source:", cluster_src)
print("Clusters scored:", len(rank_view))
print("Top CCS cluster:", rank_view.iloc[0][cluster_col] if len(rank_view) else "N/A")
print("Top CCS score:", round(float(rank_view.iloc[0]["ccs_score"]), 3) if len(rank_view) else "N/A")
print("Outputs saved to:", OUT_DIR.resolve())

print("\nTop 10 CCS clusters:")
print(
    rank_view[[
        "ccs_rank", cluster_col, "cluster_label", "risk_band",
        "ccs_score", "delay_minutes_per_vehicle", lambda_col,
        "severity_sum", "growth_pct_change", "criticality_factor"
    ]].head(10).to_string(index=False)
)

print("\nTop 10 emerging alerts:")
if len(emerging_alerts):
    print(
        emerging_alerts[[
            "alert_level", cluster_col, "cluster_label",
            "growth_pct_change", "growth_multiplier", "ccs_score", "records_total"
        ]].head(10).to_string(index=False)
    )
else:
    print("No emerging alerts found.")

# -------------------------
# Plots
# -------------------------
plt.figure(figsize=(12, 7))
top_ccs = rank_view.head(TOP_N).sort_values("ccs_score", ascending=True)
plt.barh(top_ccs["cluster_label"], top_ccs["ccs_score"])
plt.title("Top CCS Hotspots")
plt.xlabel("CCS Score")
plt.ylabel("Hotspot")
save_plot(OUT_DIR / "top_ccs_hotspots.png")

plt.figure(figsize=(12, 7))
top_delay = rank_view.head(TOP_N).sort_values("delay_minutes_per_vehicle", ascending=True)
plt.barh(top_delay["cluster_label"], top_delay["delay_minutes_per_vehicle"])
plt.title("Top Delay Hotspots")
plt.xlabel("Delay Minutes per Vehicle")
plt.ylabel("Hotspot")
save_plot(OUT_DIR / "top_delay_hotspots.png")

plt.figure(figsize=(12, 5))
plt.hist(rank_view["ccs_score"], bins=30)
plt.title("CCS Score Distribution")
plt.xlabel("CCS Score")
plt.ylabel("Clusters")
save_plot(OUT_DIR / "ccs_distribution.png")

Phase 6 complete
Input source: phase5_outputs_1/phase5_cluster_capacity_loss.csv
Clusters scored: 798
Top CCS cluster: 2
Top CCS score: 39.811
Outputs saved to: /content/phase6_outputs_1

Top 10 CCS clusters:
 ccs_rank  st_dbscan_cluster_id                                       cluster_label risk_band  ccs_score  delay_minutes_per_vehicle  lambda_hr_peak_window  severity_sum  growth_pct_change  criticality_factor
        1                     2                   JUNCTION::BTP040 - Elite Junction  Critical  39.810717               1.373741e+01                 2014.8         34509          -0.141515                 1.0
        2                     3            JUNCTION::BTP051 - Safina Plaza Junction  Critical  26.687227               4.902220e+00                 1239.2         15028          -0.184463                 1.0
        3                   128                                        CLUSTER::128  Critical  16.615964               0.000000e+00                   20.2           62

In [ ]:
import os
import zipfile
from google.colab import files

# Output zip name
zip_filename = "/content/Traffic_Intelligence_All_Phases.zip"

# Folders to include
phase_folders = [
    "/content/phase1_outputs_1",
    "/content/phase2_outputs_1",
    "/content/phase3_outputs_1",
    "/content/phase4_outputs_1",
    "/content/phase5_outputs_1",
    "/content/phase6_outputs_1",
]

# Create zip
with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:

    for folder in phase_folders:

        if os.path.exists(folder):

            for root, dirs, files_in_dir in os.walk(folder):

                for file in files_in_dir:

                    file_path = os.path.join(root, file)

                    arcname = os.path.relpath(
                        file_path,
                        start="/content"
                    )

                    zipf.write(file_path, arcname)

print(f"ZIP created successfully: {zip_filename}")

# Download automatically
files.download(zip_filename)